# Main figures — M1 to M5

One notebook, every main-text figure, in order.

| Figure | File | Panels built here |
|---|---|---|
| **Fig. 1** | `M1_rating_scatter.png` | whole figure |
| **Fig. 2** | `M2_IRFs.pdf` | a, b, c, d |
| **Fig. 3** | `M3_counterfactual.png` | a, b, c |
| **Fig. 4** | `M4_map_final.png` | a, b, c, d, e, f |
| **Fig. 5** | `M5_partA_grade.png`, `M5_partB_nonrated.png` | both parts (via `2_code/main/figures/M5_basispoint_plot.py`) |

Supplementary figures: `2_code/supplementary/figures/SI_figures1.ipynb`.

Runtime is about 4 minutes, nearly all of it section 4 — the counterfactual construction
that Figs. 2b/2d, 3 and 4 share.

## 1. Setup — imports and input paths

In [ ]:
import os
from math import exp
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
import scipy.stats as stats
from scipy.stats import norm
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, ListedColormap, BoundaryNorm
from matplotlib.ticker import FuncFormatter
from matplotlib.patches import Rectangle
import seaborn as sns
import cartopy.crs as ccrs
import geopandas as gpd
import random

plt.ion()
import os
# portable repo-root resolution (walk up to the .git / .here marker)
ROOT = os.getcwd()
while not (os.path.isdir(os.path.join(ROOT, '.git')) or os.path.isfile(os.path.join(ROOT, '.here'))) and ROOT != os.path.dirname(ROOT):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)

# Shared input paths + loaders (2_code/common/paths.py)
import sys
sys.path.insert(0, os.path.join(ROOT, '2_code', 'common'))
from paths import PATHS, coef_dir, coef_file

# Plotting + counterfactual helpers (PLOT_START_YEAR / GREY_END_YEAR are set there)
from coef_loader        import *           # noqa: F401,F403 - coefficient loading
from counterfactual import *           # noqa: F401,F403 - construction + drawing

# Output directories
OUT_DIR    = './4_figures/main/M3_counterfactuals'
M2_OUT_DIR = './4_figures/main/M2_IRF'
M4_OUT_DIR = './4_figures/main/M4_map'
SI_OUT_DIR = './4_figures/supplementary'
print(f'PLOT_START_YEAR={PLOT_START_YEAR}  GREY_END_YEAR={GREY_END_YEAR}')

# --- Fig. 1 and Fig. 2 a/c additionally need these ---------------------------
import matplotlib.gridspec as gridspec
from lp_plots import *                   # noqa: F401,F403 - IRF plotting

COEF_MAIN      = coef_dir('LP_coefficients')
BOOT_TC        = coef_file('bootstrap',      'full_public_pp_GDD_1980onwards_gl2_TC10_T02_boot_empirical_TC.csv')
BOOT_TEMP      = coef_file('bootstrap',      'full_public_pp_GDD_1980onwards_gl2_TC10_T02_boot_empirical_TEMP.csv')
BOOT_WILD_TC   = coef_file('bootstrap_wild', 'full_public_pp_GDD_1980onwards_gl2_TC10_T02_boot_wild_TC.csv')
BOOT_WILD_TEMP = coef_file('bootstrap_wild', 'full_public_pp_GDD_1980onwards_gl2_TC10_T02_boot_wild_TEMP.csv')
M1_OUT_DIR = './4_figures/main/M1_scatter'

## 2. Helper functions — imported from `2_code/common/{coef_loader,lp_plots,counterfactual}.py`

In [ ]:
# Helpers imported in the setup cell:
#   coef_loader.py         load_coefficients, extract_model_coefficients, add_zero_horizon, _append
#   lp_plots.py        plot_TC_debt (Fig. 2a), plot_TC_gdp (Fig. 2c)
#   counterfactual.py  the construction used by Figs. 2b/2d, 3 and 4, plus
#                      plot_heatmap and prepare_map_data
print('helpers loaded from coef_loader.py + lp_plots.py + counterfactual.py')

## 3. Data — every input loaded once, labelled by the figure it feeds

In [ ]:
# --- Fig. 1 ---------------------------------------------------------------
_P_RATING = PATHS['rating_df_v2']
_P_MASTER = PATHS['country_master']
_P_TEMP   = PATHS['temperature']
_P_TC     = PATHS['tc']

rating_data = pd.read_csv(_P_RATING)[['ISO', 'year', 'score_total']]
text_labels = [ # S&P and Fitch
    'CC', 'CCC-', 'CCC', 'CCC+', 'B-', 'B', 'B+', 'BB-', 'BB', 'BB+', 
    'BBB-', 'BBB', 'BBB+', 'A-', 'A', 'A+', 'AA-', 'AA', 'AA+','AAA'
]
# Load and process country master data
country_master = pd.read_csv(_P_MASTER)
sovereign_ISOs = country_master[country_master['sovereigncountry'] == 1]['ISO']

TEMP_dist = pd.read_csv(_P_TEMP)
TC_dist = pd.read_csv(_P_TC)

TEMP_2019 = TEMP_dist[TEMP_dist['year'] == 2019][['ISO', 'temp', 'temp_noANTH', 'meantemp_hist']]
TC_2019 = TC_dist[TC_dist['year'] == 2019][['ISO','avg_ws']]

TC_affected_ISOs = TC_dist[TC_dist['avg_ws']> 0.5]['ISO'].unique()

avg_warming_by_iso = TEMP_2019[['ISO', 'temp', 'temp_noANTH']].copy()
avg_warming_by_iso['warming'] = avg_warming_by_iso['temp'] - avg_warming_by_iso['temp_noANTH']
avg_warming_by_iso.drop(columns=['temp', 'temp_noANTH'], inplace=True)

TC_dist['totalexposed'] = TC_dist['ts_v2'] + TC_dist['cat1_v2'] + TC_dist['cat3_v2']
avg_TCexposure_by_iso = TC_dist[TC_dist['year'] >= 1980].groupby('ISO')['totalexposed'].mean().reset_index()

In [ ]:
# --- Fig. 2 panels a and c (LP coefficient tables) -------------------------
# TC_main / TEMP_main   -> headline LP runs + bootstrap-SE runs (Fig. 2 a/c, S5-S7)
# TC_avgws / TEMP_avgws -> average-wind-speed LP runs, term 'avg_ws' (Fig. S20 a)

# --- Fig. 2 (a, c) + Fig. S5 + Fig. S6 + Fig. S7 ---------------------------
TC_main   = load_coefficients(COEF_MAIN, shock_type='TC')
TEMP_main = load_coefficients(COEF_MAIN, shock_type='TEMP')

boot_coef_TC        = load_coefficients(BOOT_TC)
boot_coef_TEMP      = load_coefficients(BOOT_TEMP)
boot_coef_wild_TC   = load_coefficients(BOOT_WILD_TC)
boot_coef_wild_TEMP = load_coefficients(BOOT_WILD_TEMP)

TC_main   = _append(TC_main,   boot_coef_TC,        rename_map={'orig_estimate': 'estimate'})
TEMP_main = _append(TEMP_main, boot_coef_TEMP,      rename_map={'orig_estimate': 'estimate'})
TC_main   = _append(TC_main,   boot_coef_wild_TC,   rename_map={'orig_estimate': 'estimate'})
TEMP_main = _append(TEMP_main, boot_coef_wild_TEMP, rename_map={'orig_estimate': 'estimate'})

TC_main, TEMP_main = add_zero_horizon(TC_main, TEMP_main,
                                      terms=['ts_v2', 'cat1_v2', 'cat3_v2'])

# --- Fig. S20 panel a ------------------------------------------------------
TC_avgws   = load_coefficients(coef_dir('alternativeTC'), shock_type='TC')
TEMP_avgws = load_coefficients(coef_dir('alternativeTC'), shock_type='TEMP')
TC_avgws, TEMP_avgws = add_zero_horizon(TC_avgws, TEMP_avgws, terms=['avg_ws'])

print(f'TC_main   {TC_main.shape}   TEMP_main   {TEMP_main.shape}')
print(f'TC_avgws  {TC_avgws.shape}  TEMP_avgws  {TEMP_avgws.shape}')

**Fig. 2 b/d, Fig. 3, Fig. 4 — counterfactual inputs**

In [ ]:
# Load climate data
TEMP_df = pd.read_csv(PATHS['temp_data'])
TC_df = pd.read_csv(PATHS['tc_data'])
merged_df = pd.merge(TEMP_df, TC_df, on=['ISO', 'year'])

# Load country master list
country_master = pd.read_csv(PATHS['country_master'])
sovereign_ISOs = country_master[country_master['sovereigncountry'] == 1]['ISO']
sovereign_mapping = country_master[['ISO', 'sovereigncountry']].dropna()
mapping_ISOs = pd.concat([sovereign_ISOs, pd.Series(["TWN"])]).reset_index(drop=True).to_list()  # IMF GDD records Taiwan debt separately

# Load economic data
observed_gdp = pd.read_csv(PATHS['gdp_data']).rename(columns={'value': 'gdp_pcap'})
observed_debt = pd.read_csv(PATHS['debt_data']).rename(columns={'public_debt_pp': 'public_pp_gdd'})
observed = pd.merge(observed_gdp, observed_debt, on=['ISO', 'year'], how='left')

# Load coefficients
coef_debt = load_coefficients(PATHS['coef_debt'])
coef_gdp = load_coefficients(PATHS['coef_gdp'])

# Load geographic data
world = gpd.read_file(PATHS['world_shapefile'])

print(f"Data loaded: {len(merged_df)} climate obs, {len(observed)} economic obs, {len(sovereign_ISOs)} ISOs")

In [ ]:
# TC exposure panel + CMIP6 delta panel (attributable-share filters)
_tc_raw = pd.read_csv(PATHS['tc'])
_delta_src = pd.read_csv(PATHS['temp_data'])

In [ ]:
# Alternative TC counterfactual exposure datasets (raw; ISO-mapped in compute)
alternative_tc_counterfactual_files = {
    'avg_windspeed_max_intensity10pct': './1_input/counterfactual_data/avg_windspeed_max_intensity_minus10pct.csv'
}

alternative_tc_raw = {name: pd.read_csv(path)
                      for name, path in alternative_tc_counterfactual_files.items()}

In [ ]:
# SIDS membership from the shared model prep module
import sys, os
_proj_root = os.path.abspath('.')
if _proj_root not in sys.path:
    sys.path.insert(0, _proj_root)
sys.path.insert(0, os.path.join(_proj_root, '2_code', 'common'))
from prep import SIDS_ISOS as _SIDS_ISOS

In [ ]:
# ISO -> nearest TC basin mapping (SI basin boxplots)
iso_basin = pd.read_csv(PATHS['iso_to_basin'])[['ISO', 'nearest_basin']]

In [ ]:
# ENSO phase per year (Nino 3.4) — colors the ranked SI boxplots
enso = pd.read_csv(PATHS['enso_nino34'])
enso = enso[['year', 'nino34_anom_c_mean', 'enso_phase']]
phase_by_year = enso.set_index('year')['enso_phase'].to_dict()
phase_colors = {'El Nino': '#D62728', 'Neutral': '#BDBDBD', 'La Nina': '#1F77B4'}

In [ ]:
# Single-variable avg_ws LP coefficients (alternative exposure metric)
ALT_TC_DIR = coef_dir("alternativeTC") + os.sep
coef_avgws_debt = pd.read_csv(
    f"{ALT_TC_DIR}full_public_pp_GDD_unbalanced_gl2_TC10_T02_TC.csv"
)
coef_avgws_gdp = pd.read_csv(
    f"{ALT_TC_DIR}full_gdp_pcap_wdi_constant2015_USD_unbalanced_gl2_TC10_T02_TC.csv"
)

In [ ]:
map_key = pd.read_csv(PATHS['tc_isomapping'])[['ADMIN', 'ISO_for_mapping']]
ibtracs_path = PATHS.get('ibtracs_data')
if ibtracs_path and os.path.exists(ibtracs_path):
    ibtracs = pd.read_csv(ibtracs_path)
    ibtracs1 = ibtracs.iloc[1:].reset_index(drop=True)
    ibtracs1['SEASON'] = pd.to_numeric(ibtracs1['SEASON'], errors='coerce')
    filtered_ibtracs = ibtracs1[ibtracs1['SEASON'] >= 1990][['SID','NAME','LAT', 'LON', 'BASIN', 'USA_WIND', 'SEASON']]
else:
    filtered_ibtracs = pd.DataFrame(columns=['SID', 'NAME', 'LAT', 'LON', 'BASIN', 'USA_WIND', 'SEASON'])

## 4. Counterfactual construction

Shared by Figs. 2b/2d, 3 and 4 — this is the ~4 minutes.

In [ ]:
# Debt-ratio heatmap grid
grid_debt = create_heatmap_grid(coef_debt, model_type='linear', shock_type='NATH')

years_list = list(grid_debt.pivot(index="area", columns="years", values="value_ts").columns)
desired_ticks = [0, 5, 10, 15]
tick_positions = [i for i, year in enumerate(years_list) if year in desired_ticks]
tick_labels = [int(year) for year in years_list if year in desired_ticks]

ts_dist = TC_df[(TC_df['year'] >= 1980) & (TC_df['year'] < 2020)]['ts_v2'].values
cat1_dist = TC_df[(TC_df['year'] >= 1980) & (TC_df['year'] < 2020)]['cat1_v2'].values
cat3_dist = TC_df[(TC_df['year'] >= 1980) & (TC_df['year'] < 2020)]['cat3_v2'].values
sigma_value = 2.5
colors = ["#B5A8D5", "#7A73D1", "#4D55CC"]

In [ ]:
# GDP-per-capita heatmap grid (sign flipped so damages plot positive)
grid_gdp = create_heatmap_grid(coef_gdp, model_type='linear', shock_type='NATH', sign=-1)

In [ ]:
# Extract coefficients
coefs_debt = extract_model_coefficients(coef_debt)
coefs_gdp = extract_model_coefficients(coef_gdp)

In [ ]:
# PROCESS coefficients (DEBT RATIO)
results_TC = generate_iso_yearly_vectors_onlyTC(merged_df, coefs_debt['ts'], coefs_debt['cat1'], coefs_debt['cat3'])
results_df_onlyTC = process_results(results_TC)

results_TEMP = generate_iso_yearly_vectors_onlyTEMP_delta(merged_df, coefs_debt['temp'], coefs_debt['temp_interaction'])
results_df_onlyTEMP_delta = process_results(results_TEMP)

# PROCESS coefficients (GDP PER CAPITA)
results_TC_gdp = generate_iso_yearly_vectors_onlyTC(merged_df, coefs_gdp['ts'], coefs_gdp['cat1'], coefs_gdp['cat3'])
results_df_onlyTC_gdp = process_results(results_TC_gdp)

results_TEMP_gdp = generate_iso_yearly_vectors_onlyTEMP_delta(merged_df, coefs_gdp['temp'], coefs_gdp['temp_interaction'])
results_df_onlyTEMP_delta_gdp = process_results(results_TEMP_gdp)

In [ ]:
# CALCULATE COUNTERFACTUALS
select_countries = ["USA", "PHL", 'BRB']
combined_TC = calculate_country_counterfactuals(select_countries, observed_debt, results_df_onlyTC, 'public_pp_gdd')
gdp_combined_cf = calculate_combined_counterfactuals(select_countries, observed, results_df_onlyTC_gdp, results_df_onlyTEMP_delta_gdp, 'gdp_pcap')

In [ ]:
# === Extended time series ===

# === debt-ratio time series ===
results_TC_ext        = generate_iso_yearly_vectors_onlyTC(
                            merged_df, coefs_debt['ts'], coefs_debt['cat1'], coefs_debt['cat3'],
                            start_year=PLOT_START_YEAR)
results_df_onlyTC_ext = process_results(results_TC_ext, start_year=PLOT_START_YEAR)

# === GDP time series ===
results_TC_gdp_ext        = generate_iso_yearly_vectors_onlyTC(
                                merged_df, coefs_gdp['ts'], coefs_gdp['cat1'], coefs_gdp['cat3'],
                                start_year=PLOT_START_YEAR)
results_TEMP_gdp_ext              = generate_iso_yearly_vectors_onlyTEMP_delta(
                                        merged_df, coefs_gdp['temp'], coefs_gdp['temp_interaction'],
                                        start_year=PLOT_START_YEAR)
results_df_onlyTC_gdp_ext = process_results(results_TC_gdp_ext, start_year=PLOT_START_YEAR)
results_df_onlyTEMP_delta_gdp_ext = process_results(results_TEMP_gdp_ext, start_year=PLOT_START_YEAR)

In [ ]:
select_countries = ['USA', 'PHL', 'BRB']

In [ ]:
# --- DEBT RATIO extended counterfactual stack ---
combined_TC_ext       = calculate_country_counterfactuals(
                            select_countries, 
                            observed_debt, results_df_onlyTC_ext,
                            'public_pp_gdd',
                            start_year=PLOT_START_YEAR, end_year=2022)

In [ ]:
# --- GDP extended counterfactual (TC + temp combined) ---
gdp_combined_cf_ext = calculate_combined_counterfactuals(
                          select_countries, observed,
                          results_df_onlyTC_gdp_ext, results_df_onlyTEMP_delta_gdp_ext,
                          'gdp_pcap',
                          start_year=PLOT_START_YEAR)

In [ ]:
# Share of observed debt / GDP attributable to TCs and temperature, per country:
# (observed - counterfactual) / observed, 1980-2019. Debt: TC only; GDP: TC, temp and combined.

# Counterfactuals for all countries (1970 start).
all_isos_debt = sorted(results_df_onlyTC_ext['ISO'].unique())
# GDP: every country with a TC or a temperature counterfactual.
all_isos_gdp  = sorted(set(results_df_onlyTC_gdp_ext['ISO']) |
                       set(results_df_onlyTEMP_delta_gdp_ext['ISO']))

combined_TC_all = calculate_country_counterfactuals(
    all_isos_debt, observed_debt, results_df_onlyTC_ext,
    'public_pp_gdd', start_year=PLOT_START_YEAR, end_year=2022,
)
gdp_combined_all = calculate_combined_counterfactuals(
    all_isos_gdp, observed,
    results_df_onlyTC_gdp_ext, results_df_onlyTEMP_delta_gdp_ext,
    'gdp_pcap', start_year=PLOT_START_YEAR,
)

# Extratropical countries: entirely north of 23.5N or south of 23.5S.
_country_bounds = (
    world[['ADM0_A3', 'geometry']]
    .dropna(subset=['ADM0_A3'])
    .assign(_miny=lambda d: d.geometry.bounds['miny'],
            _maxy=lambda d: d.geometry.bounds['maxy'])
    .groupby('ADM0_A3')
    .agg(miny=('_miny', 'min'), maxy=('_maxy', 'max'))
)
extratropical_isos = set(
    _country_bounds.index[(_country_bounds['miny'] >  23.5) |
                           (_country_bounds['maxy'] < -23.5)]
)
# NOR added by hand (Bouvet Island pulls its bounds south).
extratropical_isos |= {'NOR'}
print(f'extratropical_isos: {len(extratropical_isos)} countries')

# Attributable shares, 1980-2019.
mask_debt = (combined_TC_all['year'] >= 1980) & (combined_TC_all['year'] <= 2019)
combined_TC_all['share_attributable'] = np.where(
    mask_debt,
    (combined_TC_all['public_pp_gdd'] - combined_TC_all[f'debt_{PLOT_START_YEAR}'])
        / combined_TC_all['public_pp_gdd'],
    np.nan,
)

# GDP TC and combined: (obs - cf) / obs (negative = damage); temp: (cf - obs) / cf.
mask_gdp = (gdp_combined_all['year'] >= 1980) & (gdp_combined_all['year'] <= 2019)
for suffix in ['TC', 'combined']:
    cf_col = f'{PLOT_START_YEAR}_cf_{suffix}'
    gdp_combined_all[f'share_attributable_{suffix}'] = np.where(
        mask_gdp,
        (gdp_combined_all['gdp_pcap'] - gdp_combined_all[cf_col])
            / gdp_combined_all['gdp_pcap'],
        np.nan,
    )
_cf_temp = f'{PLOT_START_YEAR}_cf_temp'
gdp_combined_all['share_attributable_temp'] = np.where(
    mask_gdp,
    (gdp_combined_all[_cf_temp] - gdp_combined_all['gdp_pcap'])
        / gdp_combined_all[_cf_temp],
    np.nan,
)
gdp_combined_all['share_attributable'] = gdp_combined_all['share_attributable_combined']

# Countries with at least one year of TS-or-stronger exposure, 1980-2019.
_tc_raw = pd.read_csv(PATHS['tc'])
_tc_panel = _tc_raw[(_tc_raw['year']>=1980) & (_tc_raw['year']<=2019)].copy()
_tc_panel['_shock'] = ((_tc_panel['ts_v2']>0) |
                        (_tc_panel['cat1_v2']>0) |
                        (_tc_panel['cat3_v2']>0))
valid_shock_isos = set(_tc_panel.groupby('ISO')['_shock'].sum()
                                  .pipe(lambda s: s[s>=1].index))
print(f'valid_shock_isos: {len(valid_shock_isos)} ISOs with >=1 shock year (1980-2019)')

# Countries with CMIP6 warming data (delta) in 1980-2019; the rest have no temperature counterfactual.
_delta_src = pd.read_csv(PATHS['temp_data'])
_delta_panel = _delta_src[(_delta_src['year']>=1980) &
                            (_delta_src['year']<=2019)]
valid_delta_isos = set(_delta_panel.groupby('ISO')['delta']
                                      .apply(lambda s: s.notna().any())
                                      .pipe(lambda s: s[s].index))
print(f'valid_delta_isos: {len(valid_delta_isos)} ISOs with >=1 valid delta year (1980-2019)')

# Per-country median and IQR (sovereigns only).
sov_set = set(sovereign_ISOs)

def country_stats(df, value_col='share_attributable', sort_ascending=True,
                  require_shock=True):
    """Per-country median + IQR. Filters: sovereign, valid CMIP6 attribution
    data (non-NaN delta), n>=5 valid years, and (when require_shock=True) ISO
    has >=1 above-TS shock year in 1980-2019."""
    g = (df[df['ISO'].isin(sov_set) & df['ISO'].isin(valid_delta_isos)]
           .dropna(subset=[value_col])
           .groupby('ISO')
           .agg(median=(value_col, 'median'),
                q25=(value_col, lambda s: s.quantile(0.25)),
                q75=(value_col, lambda s: s.quantile(0.75)),
                n=(value_col, 'count'))
           .reset_index())
    g = g[g['n'] >= 5].copy()
    if require_shock:
        g = g[g['ISO'].isin(valid_shock_isos)].copy()
    g['is_extra'] = g['ISO'].isin(extratropical_isos)
    g = g.sort_values(['is_extra', 'median'], ascending=[False, sort_ascending])
    return g.reset_index(drop=True)

debt_stats     = country_stats(combined_TC_all,                                   sort_ascending=True)
gdp_stats_TC   = country_stats(gdp_combined_all, value_col='share_attributable_TC',   sort_ascending=False)
# GDP temp: all countries, including those without TC exposure.
gdp_stats_temp = country_stats(gdp_combined_all, value_col='share_attributable_temp',
                                sort_ascending=False, require_shock=False)

# Negate the temp share for display so climate damage is negative, as in the TC panels.
gdp_stats_temp = gdp_stats_temp.copy()
_old_q25 = gdp_stats_temp['q25'].copy()
gdp_stats_temp['median'] = -gdp_stats_temp['median']
gdp_stats_temp['q25']    = -gdp_stats_temp['q75']
gdp_stats_temp['q75']    = -_old_q25

# GDP combined TC + temp: all countries.
gdp_stats = country_stats(gdp_combined_all,
                          value_col='share_attributable',
                          sort_ascending=False, require_shock=False)

# SI panels: GDP TC and temp shares for TC-exposed countries (extratropical listed first, drawn at the bottom).
gdp_TC_main  = (gdp_stats_TC[~gdp_stats_TC['is_extra']]
                 .sort_values('median', ascending=False)
                 .reset_index(drop=True))
gdp_TC_extra = (gdp_stats_TC[ gdp_stats_TC['is_extra']]
                 .sort_values('median', ascending=False)
                 .reset_index(drop=True))

plot2_tc = pd.concat(
    [gdp_TC_extra[['ISO','median','q25','q75']],
     gdp_TC_main [['ISO','median','q25','q75']]],
    ignore_index=True,
)

temp_lookup = gdp_stats_temp.set_index('ISO')[['median','q25','q75']]
plot2_temp = (temp_lookup
                .reindex(plot2_tc['ISO'].values)
                .reset_index()
                .rename(columns={'index':'ISO'}))

is_extra_arr = ([True] * len(gdp_TC_extra)) + ([False] * len(gdp_TC_main))

# SI panels: GDP temp share for countries without TC exposure, split over two columns.
isos_in_plot2 = set(plot2_tc['ISO'])
gdp_temp_other = (gdp_stats_temp[~gdp_stats_temp['ISO'].isin(isos_in_plot2)]
                   .copy())
gdp_temp_other = gdp_temp_other[gdp_temp_other['median'].abs() > 1e-4].copy()
gdp_temp_other = (gdp_temp_other
                   .sort_values('median', ascending=True)
                   .reset_index(drop=True))

n = len(gdp_temp_other)
half = (n + 1) // 2
gdp_temp_left  = gdp_temp_other.iloc[:half].reset_index(drop=True)
gdp_temp_right = gdp_temp_other.iloc[half:].reset_index(drop=True)

In [ ]:
# attributable_shares.csv: per-country median shares and 2019 observed / counterfactual levels.
# gdp_share_* and gdp_pcap_cf_2019 are the combined TC + temp impact.
# Read by generate_XGB_predictions.py, M5_basispoint_plot.py and credit_rating_impacts.py.
debt_save = debt_stats[['ISO', 'median', 'q25', 'q75', 'n']].rename(
    columns={
        'median': 'debt_share_median',
        'q25':    'debt_share_q25',
        'q75':    'debt_share_q75',
        'n':      'debt_share_n',
    }
)
def _gdp_save(stats_df, prefix):
    return stats_df[['ISO', 'median', 'q25', 'q75', 'n']].rename(
        columns={
            'median': f'{prefix}_median',
            'q25':    f'{prefix}_q25',
            'q75':    f'{prefix}_q75',
            'n':      f'{prefix}_n',
        }
    )
gdp_save_combined = _gdp_save(gdp_stats,      'gdp_share')
gdp_save_TC       = _gdp_save(gdp_stats_TC,   'gdp_share_TC')
gdp_save_temp     = _gdp_save(gdp_stats_temp, 'gdp_share_temp')

attributable_shares = (
    debt_save
    .merge(gdp_save_combined, on='ISO', how='outer')
    .merge(gdp_save_TC,       on='ISO', how='outer')
    .merge(gdp_save_temp,     on='ISO', how='outer')
)

# 2019 observed and counterfactual levels.
debt_2019 = (
    combined_TC_all[combined_TC_all['year'] == 2019]
    [['ISO', 'public_pp_gdd', f'debt_{PLOT_START_YEAR}']]
    .rename(columns={
        'public_pp_gdd':           'debt_observed_2019',
        f'debt_{PLOT_START_YEAR}': 'debt_cf_2019',
    })
)
gdp_2019 = (
    gdp_combined_all[gdp_combined_all['year'] == 2019]
    [['ISO', 'gdp_pcap',
      f'{PLOT_START_YEAR}_cf_TC',
      f'{PLOT_START_YEAR}_cf_temp',
      f'{PLOT_START_YEAR}_cf_combined']]
    .rename(columns={
        'gdp_pcap':                       'gdp_pcap_observed_2019',
        f'{PLOT_START_YEAR}_cf_TC':       'gdp_pcap_cf_TC_2019',
        f'{PLOT_START_YEAR}_cf_temp':     'gdp_pcap_cf_temp_2019',
        f'{PLOT_START_YEAR}_cf_combined': 'gdp_pcap_cf_combined_2019',
    })
)
gdp_2019['gdp_pcap_cf_2019'] = gdp_2019['gdp_pcap_cf_combined_2019']

attributable_shares = (
    attributable_shares
    .merge(debt_2019, on='ISO', how='left')
    .merge(gdp_2019,  on='ISO', how='left')
)

attributable_shares['is_extratropical'] = (
    attributable_shares['ISO'].isin(extratropical_isos)
)

out_path = './3_output/counterfactual_data/attributable_shares.csv'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
attributable_shares.to_csv(out_path, index=False)
print(
    f'Saved {len(attributable_shares)} sovereigns to {out_path}\n'
    f'  {(~attributable_shares["is_extratropical"]).sum()} tropical, '
    f'{attributable_shares["is_extratropical"].sum()} extratropical\n'
    f'  Columns: {list(attributable_shares.columns)}'
)

attributable_shares.head()

In [ ]:
# Convert yearly TC counterfactual vectors to a long DataFrame for alternative scenarios
analysis_years = list(range(1990, 2020))
vector_length = len(analysis_years)

results_TC_vectors = yearly_vectors_to_long(results_TC)

tc_maxws_complete = TC_df
tc_maxws_complete = tc_maxws_complete[(tc_maxws_complete['year'] >= 1990) & (tc_maxws_complete['year'] <= 2019)].copy()

debt_lookup = {
    iso: grp.set_index('year').reindex(analysis_years)['public_pp_gdd'].to_numpy(dtype=float)
    for iso, grp in observed_debt[
        (observed_debt['year'] >= 1990) & (observed_debt['year'] <= 2019)
    ][['ISO', 'year', 'public_pp_gdd']].groupby('ISO')
}

results_TC_vectors_merged = build_tc_vectors_merged(results_TC, tc_maxws_complete, debt_lookup)

alternative_tc_exposure_data = {}
alternative_results_TC = {}
alternative_results_TC_vectors_merged = {}
alternative_tc_unmatched_countries = {}

for scenario_name in alternative_tc_counterfactual_files:
    alt_tc_df, unmatched_countries = prepare_alternative_tc_exposure(alternative_tc_raw[scenario_name], country_master)
    alt_results_TC = generate_iso_yearly_vectors_onlyTC(alt_tc_df, coefs_debt['ts'], coefs_debt['cat1'], coefs_debt['cat3'])
    alternative_tc_exposure_data[scenario_name] = alt_tc_df
    alternative_results_TC[scenario_name] = alt_results_TC
    alternative_results_TC_vectors_merged[scenario_name] = build_tc_vectors_merged(alt_results_TC, alt_tc_df, debt_lookup)
    alternative_tc_unmatched_countries[scenario_name] = unmatched_countries

results_TC_vectors_merged_avg_windspeed_max_intensity10pct = alternative_results_TC_vectors_merged['avg_windspeed_max_intensity10pct']

pd.DataFrame([
    {
        'scenario': 'baseline_tc_maxws_complete',
        'n_rows': len(results_TC_vectors_merged),
        'n_isos': results_TC_vectors_merged['ISO'].nunique(),
        'n_unmatched_countries': 0
    },
    *[
        {
            'scenario': scenario_name,
            'n_rows': len(alternative_results_TC_vectors_merged[scenario_name]),
            'n_isos': alternative_results_TC_vectors_merged[scenario_name]['ISO'].nunique(),
            'n_unmatched_countries': len(alternative_tc_unmatched_countries[scenario_name])
        }
        for scenario_name in alternative_tc_counterfactual_files
    ]
])

In [ ]:
# Alternative severity definitions
top5_totalexposed_removed, cumulative_impact_top5_totalexposed_removed, top5_totalexposed_removed_paths = build_alternative_counterfactual(
    results_TC_vectors_merged, 'totalexposed', 5, debt_lookup
)
top10_totalexposed_removed, cumulative_impact_top10_totalexposed_removed, top10_totalexposed_removed_paths = build_alternative_counterfactual(
    results_TC_vectors_merged, 'totalexposed', 10, debt_lookup
)
top5_avg_ws_removed, cumulative_impact_top5_avg_ws_removed, top5_avg_ws_removed_paths = build_alternative_counterfactual(
    results_TC_vectors_merged, 'avg_ws', 5, debt_lookup
)
top10_avg_ws_removed, cumulative_impact_top10_avg_ws_removed, top10_avg_ws_removed_paths = build_alternative_counterfactual(
    results_TC_vectors_merged, 'avg_ws', 10, debt_lookup
)

alternative_counterfactuals = {
    'top5_totalexposed_removed': cumulative_impact_top5_totalexposed_removed,
    'top10_totalexposed_removed': cumulative_impact_top10_totalexposed_removed,
    'top5_avg_ws_removed': cumulative_impact_top5_avg_ws_removed,
    'top10_avg_ws_removed': cumulative_impact_top10_avg_ws_removed
}
alternative_counterfactual_paths = {
    'top5_totalexposed_removed': top5_totalexposed_removed_paths,
    'top10_totalexposed_removed': top10_totalexposed_removed_paths,
    'top5_avg_ws_removed': top5_avg_ws_removed_paths,
    'top10_avg_ws_removed': top10_avg_ws_removed_paths
}

pd.DataFrame({
    'scenario': list(alternative_counterfactuals.keys()),
    'n_countries': [df['ISO'].nunique() for df in alternative_counterfactuals.values()]
})

In [ ]:
# Compare 2019 debt-ratio impacts across alternate TC counterfactual datasets
alt_select_countries = ['USA','PHL','BRB','DOM']

alt_titles = ['United States', 'The Philippines','Barbados','Dominican Republic', 'Dominica', 'Jamaica']


alternative_results_df_onlyTC = {
    scenario_name: process_results(results_dict)
    for scenario_name, results_dict in alternative_results_TC.items()
}

# Scenarios in display order. (label, kind, source)
# kind='alt'   uses alternative_results_df_onlyTC[source]
# kind='paths' uses a *_removed_paths dataframe directly (debt_1990 at year=2019)
# kind='base'  is the all-TC-removed baseline from results_df_onlyTC
alt_plot_scenarios = [
    ('all TC removed',             'base',   None),
    ('10% reduced intensity',      'alt',    'avg_windspeed_max_intensity10pct'),
    ('Top 5 storm years removed',  'paths',  top5_avg_ws_removed_paths),
    ('Top 10 storm years removed', 'paths',  top10_avg_ws_removed_paths),
]


impact_rows = []
for label, kind, source in alt_plot_scenarios:
    if kind == 'base':
        df_ = impact_for_base(alt_select_countries, observed_debt, results_df_onlyTC)
    elif kind == 'alt':
        df_ = impact_for_alt(source, alt_select_countries, observed_debt, alternative_results_df_onlyTC)
    else:
        df_ = impact_for_paths(source, alt_select_countries, observed_debt)
    df_['scenario'] = label
    impact_rows.append(df_)
alternative_impact_2019 = pd.concat(impact_rows, ignore_index=True)

# Row 2: % of all-TC-removed (impact remaining after the modification)
base_by_iso = (alternative_impact_2019[alternative_impact_2019['scenario'] == 'all TC removed']
               .set_index('ISO')['impact_2019'])
alternative_impact_2019['pct_of_all_tc'] = alternative_impact_2019.apply(
    lambda r: r['impact_2019'] / base_by_iso[r['ISO']] * 100 if base_by_iso[r['ISO']] != 0 else np.nan, axis=1
)
# Row 3 (pp): pp attributable to the modified subset = base_impact_pp - scenario_impact_pp.
# For 'all TC removed' the attributable bar is shown at its full base value (i.e., the total TC impact).
alternative_impact_2019['impact_attributable_pp'] = np.where(
    alternative_impact_2019['scenario'] == 'all TC removed',
    alternative_impact_2019['ISO'].map(base_by_iso),
    alternative_impact_2019['ISO'].map(base_by_iso) - alternative_impact_2019['impact_2019'],
)
# Row 4 (%): share of all-TC-removed impact attributable to the modified subset.
# 'all TC removed' is shown at 100% (the full total).
alternative_impact_2019['pct_attributable'] = np.where(
    alternative_impact_2019['scenario'] == 'all TC removed',
    100,
    100 - alternative_impact_2019['pct_of_all_tc'],
)

In [ ]:
# === Regional counterfactual plot ===

label_remap_r = {
    'all TC removed':              'all TCs',
    'Top 10 storm years removed':  'Top 10\nseasons',
    'Top 5 storm years removed':   'Top 5\nseasons',
    '10% reduced intensity':       '10%\nintensity\nreduced',
}
# Column order (Fig. 4f): all TCs, 10% intensity reduced, Top 10 seasons, Top 5 seasons;
# `subset_colors_r` follows the same order.
bar_subset_order_r = ['all TCs', '10%\nintensity\nreduced', 'Top 10\nseasons', 'Top 5\nseasons']
subset_colors_r = ['#D9D9D9', '#8C6239', '#CC6600', '#F4B86A']

scenarios_r = [s for s in alt_plot_scenarios if s[0] in label_remap_r]

# --- Build per-country impact across the 4 scenarios for ALL sovereigns ---
all_isos_r = sovereign_ISOs.tolist()

rows_r = []
for label, kind, source in scenarios_r:
    if kind == 'base':
        cf = calculate_country_counterfactuals(all_isos_r, observed_debt, results_df_onlyTC, 'public_pp_gdd')
        df_ = cf[cf['year'] == 2019][['ISO', 'public_pp_gdd', 'debt_1990']].copy()
    elif kind == 'alt':
        cf = calculate_country_counterfactuals(all_isos_r, observed_debt, alternative_results_df_onlyTC[source], 'public_pp_gdd')
        df_ = cf[cf['year'] == 2019][['ISO', 'public_pp_gdd', 'debt_1990']].copy()
    else:
        obs = observed_debt[(observed_debt['year'] == 2019) & (observed_debt['ISO'].isin(all_isos_r))][['ISO', 'public_pp_gdd']]
        df_ = source[source['year'] == 2019][['ISO', 'debt_1990']].merge(obs, on='ISO', how='inner')
    df_['impact_2019'] = df_['public_pp_gdd'] - df_['debt_1990']
    df_['scenario'] = label_remap_r[label]
    rows_r.append(df_)
all_impact_r = pd.concat(rows_r, ignore_index=True)

# Restrict to non-zero baseline ISOs and attach baseline + per-country attributable columns
base_by_iso_r = all_impact_r[all_impact_r['scenario'] == 'all TCs'].set_index('ISO')['impact_2019']
all_impact_r = all_impact_r[all_impact_r['ISO'].isin(base_by_iso_r[base_by_iso_r.abs() > 1e-6].index)].copy()
all_impact_r['base_iso'] = all_impact_r['ISO'].map(base_by_iso_r)

denom_r = all_impact_r['public_pp_gdd'] - all_impact_r['impact_2019']
all_impact_r['rate_iso'] = np.where(denom_r.abs() > 1e-9, all_impact_r['impact_2019'] / denom_r, np.nan)
base_rate_by_iso_r = all_impact_r[all_impact_r['scenario'] == 'all TCs'].set_index('ISO')['rate_iso']
all_impact_r['base_rate_iso'] = all_impact_r['ISO'].map(base_rate_by_iso_r)

all_impact_r['rate_attributable_country'] = np.where(
    all_impact_r['scenario'] == 'all TCs',
    all_impact_r['base_rate_iso'],
    all_impact_r['base_rate_iso'] - all_impact_r['rate_iso'],
)
all_impact_r['pct_attributable_country'] = np.where(
    all_impact_r['scenario'] == 'all TCs',
    100,
    np.where(all_impact_r['base_iso'].abs() > 1e-9,
             (all_impact_r['base_iso'] - all_impact_r['impact_2019']) / all_impact_r['base_iso'] * 100,
             np.nan),
)

# Attach Region grouping + SIDS membership
sids_col = 'Small.Island.Developing.States..SIDS.'
country_groups_r = (country_master[['ISO', 'Region.Name', sids_col]]
                    .drop_duplicates('ISO'))
all_impact_r = all_impact_r.merge(country_groups_r, on='ISO', how='left')
all_impact_r['region'] = all_impact_r['Region.Name'].fillna('Unknown')
all_impact_r['is_sids'] = all_impact_r[sids_col].fillna(0).astype(int)

# Aggregate per region (n_countries used for panel titles)
agg_r = (all_impact_r.groupby(['region', 'scenario'], as_index=False)
         .agg(n_countries=('ISO', 'nunique')))
# Append synthetic blocks: Global (all), SIDS (is_sids==1), non-SIDS (is_sids==0)
global_block = (all_impact_r.assign(region='Global').groupby(['region', 'scenario'], as_index=False)
                .agg(n_countries=('ISO', 'nunique')))
sids_block   = (all_impact_r[all_impact_r['is_sids'] == 1].assign(region='SIDS')
                .groupby(['region', 'scenario'], as_index=False)
                .agg(n_countries=('ISO', 'nunique')))
nonsids_block = (all_impact_r[all_impact_r['is_sids'] == 0].assign(region='non-SIDS')
                 .groupby(['region', 'scenario'], as_index=False)
                 .agg(n_countries=('ISO', 'nunique')))
agg_r = pd.concat([global_block, sids_block, nonsids_block, agg_r], ignore_index=True)

# Display order: Global, SIDS, non-SIDS, Americas, Asia, Oceania (no Africa / Europe).
region_order_r = ['Global', 'SIDS', 'non-SIDS', 'Americas', 'Asia', 'Oceania']
region_order_r = [g for g in region_order_r if g in set(agg_r['region'])]

In [ ]:
# === Regional counterfactual plot — hybrid metric per scenario ===
# Y-axis: rate (x100), per scenario:
#   all TCs                        -> base_rate_iso (full TC rate)
#   10% reduced intensity          -> rate_iso
#   Top 10 / Top 5 seasons removed -> base_rate_iso - rate_iso (rate attributable to those seasons)


label_remap_v = {
    'all TC removed':              'all TCs',
    '10% reduced intensity':       '10%\nintensity\nreduced',
    'Top 10 storm years removed':  'Top 10\nseasons',
    'Top 5 storm years removed':   'Top 5\nseasons',
}
# Display order: No TCs (full removal) → 10% intensity → Top 10 → Top 5.
bar_subset_order_v = ['all TCs', '10%\nintensity\nreduced',
                       'Top 10\nseasons', 'Top 5\nseasons']
subset_colors_v = ['#D9D9D9', '#8C6239', '#CC6600', '#F4B86A']

scenarios_v = [s for s in alt_plot_scenarios if s[0] in label_remap_v]
all_isos_v = sovereign_ISOs.tolist()

rows_v = []
for label, kind, source in scenarios_v:
    if kind == 'base':
        cf = calculate_country_counterfactuals(all_isos_v, observed_debt,
                                                 results_df_onlyTC, 'public_pp_gdd')
        df_ = cf[cf['year'] == 2019][['ISO', 'public_pp_gdd', 'debt_1990']].copy()
    elif kind == 'alt':
        cf = calculate_country_counterfactuals(all_isos_v, observed_debt,
                                                 alternative_results_df_onlyTC[source],
                                                 'public_pp_gdd')
        df_ = cf[cf['year'] == 2019][['ISO', 'public_pp_gdd', 'debt_1990']].copy()
    else:
        obs = (observed_debt[(observed_debt['year'] == 2019)
                              & (observed_debt['ISO'].isin(all_isos_v))]
               [['ISO', 'public_pp_gdd']])
        df_ = source[source['year'] == 2019][['ISO', 'debt_1990']].merge(obs, on='ISO', how='inner')
    df_['impact_2019'] = df_['public_pp_gdd'] - df_['debt_1990']
    df_['scenario']    = label_remap_v[label]
    rows_v.append(df_)
all_impact_v = pd.concat(rows_v, ignore_index=True)

# Restrict to ISOs whose 'all TCs' baseline is non-zero.
base_v = all_impact_v[all_impact_v['scenario'] == 'all TCs'].set_index('ISO')['impact_2019']
all_impact_v = all_impact_v[all_impact_v['ISO'].isin(base_v[base_v.abs() > 1e-6].index)].copy()

# rate_iso = impact_2019 / cf_debt_2019 (cf_debt_2019 = debt_1990).
all_impact_v['cf_debt_2019'] = all_impact_v['public_pp_gdd'] - all_impact_v['impact_2019']
all_impact_v['rate_iso'] = np.where(
    all_impact_v['cf_debt_2019'].abs() > 1e-9,
    all_impact_v['impact_2019'] / all_impact_v['cf_debt_2019'],
    np.nan,
)
# Per-ISO base_rate (rate_iso under the 'all TCs removed' scenario).
_base_rate = (all_impact_v[all_impact_v['scenario'] == 'all TCs']
                .set_index('ISO')['rate_iso'])
all_impact_v['base_rate_iso'] = all_impact_v['ISO'].map(_base_rate)

# Plot metric per scenario (see top of cell).
_PARTIAL_BASE_MINUS = {'Top 10\nseasons', 'Top 5\nseasons'}
all_impact_v['plot_rate'] = np.where(
    all_impact_v['scenario'] == 'all TCs',
    all_impact_v['base_rate_iso'],
    np.where(
        all_impact_v['scenario'].isin(_PARTIAL_BASE_MINUS),
        all_impact_v['base_rate_iso'] - all_impact_v['rate_iso'],
        all_impact_v['rate_iso'],
    ),
)

# Region + SIDS membership.
_iso_region = (country_master[['ISO', 'Region.Name']].drop_duplicates('ISO')
                if 'Region.Name' in country_master.columns
                else pd.DataFrame({'ISO': all_isos_v, 'Region.Name': 'Unknown'}))
all_impact_v = all_impact_v.merge(_iso_region, on='ISO', how='left')
all_impact_v['region']  = all_impact_v['Region.Name'].fillna('Unknown')
all_impact_v['is_sids'] = all_impact_v['ISO'].isin(set(_SIDS_ISOS)).astype(int)

In [ ]:
# Attach SIDS flag to the existing per-country impact frame.
# (drop any existing column first to avoid _x/_y suffixes)
_sids_col = 'Small.Island.Developing.States..SIDS.'
sids_lookup = country_master[['ISO', _sids_col]].drop_duplicates('ISO')
all_impact_ext = (all_impact_r.drop(columns=[_sids_col], errors='ignore')
                              .merge(sids_lookup, on='ISO', how='left'))
all_impact_ext['sids_status'] = np.where(
    all_impact_ext[_sids_col] == 1, 'SIDS', 'Non-SIDS'
)

In [ ]:
# === SUPPLEMENTARY BOXPLOTS — hybrid metric ===
# Two plots: (1) Aggregated by TC basin, (2) Aggregated by Region.
# Row 1: rate metric; row 2: % of total impact (all TCs = 100; 10% reduced = share retained;
# Top 10 / Top 5 = share attributable to those seasons).
# Region plot adds SIDS and non-SIDS panels; Europe becomes "Extratropical" and pools
# all extratropical sovereigns.


label_remap = {
    'all TC removed':              'all TCs',
    '10% reduced intensity':       '10%\nintensity\nreduced',
    'Top 10 storm years removed':  'Top 10\nseasons',
    'Top 5 storm years removed':   'Top 5\nseasons',
}
bar_subset_order = ['all TCs', '10%\nintensity\nreduced', 'Top 10\nseasons', 'Top 5\nseasons']
subset_colors = ['#D9D9D9', '#8C6239', '#CC6600', '#F4B86A']

# --- Compute impact per (ISO, scenario) for ALL sovereigns ---
selected_scenarios = [s for s in alt_plot_scenarios if s[0] in label_remap]
all_isos = sovereign_ISOs.tolist()
impact_rows_all = []
for label, kind, source in selected_scenarios:
    if kind == 'base':
        cf = calculate_country_counterfactuals(all_isos, observed_debt,
                                                results_df_onlyTC, 'public_pp_gdd')
        df_ = cf[cf['year'] == 2019][['ISO', 'public_pp_gdd', 'debt_1990']].copy()
    elif kind == 'alt':
        cf = calculate_country_counterfactuals(all_isos, observed_debt,
                                                alternative_results_df_onlyTC[source],
                                                'public_pp_gdd')
        df_ = cf[cf['year'] == 2019][['ISO', 'public_pp_gdd', 'debt_1990']].copy()
    else:
        obs = (observed_debt[(observed_debt['year'] == 2019)
                              & (observed_debt['ISO'].isin(all_isos))]
               [['ISO', 'public_pp_gdd']])
        df_ = source[source['year'] == 2019][['ISO', 'debt_1990']].merge(obs, on='ISO', how='inner')
    df_['impact_2019'] = df_['public_pp_gdd'] - df_['debt_1990']
    df_['scenario'] = label_remap[label]
    impact_rows_all.append(df_)
all_impact = pd.concat(impact_rows_all, ignore_index=True)

# Restrict to ISOs whose 'all TCs' baseline impact is non-zero.
base_by_iso_all = all_impact[all_impact['scenario'] == 'all TCs'].set_index('ISO')['impact_2019']
nonzero_isos = base_by_iso_all[base_by_iso_all.abs() > 1e-6].index
all_impact = all_impact[all_impact['ISO'].isin(nonzero_isos)].copy()
all_impact['base_iso'] = all_impact['ISO'].map(base_by_iso_all)

# Per-scenario rate_iso = impact_2019 / cf_2019 = impact_2019 / (obs - impact_2019).
denom = all_impact['public_pp_gdd'] - all_impact['impact_2019']
all_impact['rate_iso'] = np.where(denom.abs() > 1e-9,
                                    all_impact['impact_2019'] / denom, np.nan)
base_rate_by_iso = all_impact[all_impact['scenario'] == 'all TCs'].set_index('ISO')['rate_iso']
all_impact['base_rate_iso'] = all_impact['ISO'].map(base_rate_by_iso)

# Hybrid metric for row 1 (rate).
_PARTIAL_BASE_MINUS = {'Top 10\nseasons', 'Top 5\nseasons'}
all_impact['plot_rate'] = np.where(
    all_impact['scenario'] == 'all TCs',
    all_impact['base_rate_iso'],
    np.where(
        all_impact['scenario'].isin(_PARTIAL_BASE_MINUS),
        all_impact['base_rate_iso'] - all_impact['rate_iso'],
        all_impact['rate_iso'],  # 10% intensity reduced
    ),
)

# Hybrid metric for row 2 (% of total impact).
all_impact['plot_pct'] = np.where(
    all_impact['scenario'] == 'all TCs',
    100.0,
    np.where(
        all_impact['scenario'] == '10%\nintensity\nreduced',
        np.where(all_impact['base_iso'].abs() > 1e-9,
                  all_impact['impact_2019'] / all_impact['base_iso'] * 100,
                  np.nan),
        np.where(all_impact['base_iso'].abs() > 1e-9,
                  (all_impact['base_iso'] - all_impact['impact_2019'])
                  / all_impact['base_iso'] * 100, np.nan),
    ),
)

# Attach grouping columns
country_groups = (country_master[['ISO', 'Region.Name',
                                    'Small.Island.Developing.States..SIDS.']]
                   .drop_duplicates('ISO'))
all_impact = (all_impact
              .merge(iso_basin, on='ISO', how='left')
              .merge(country_groups, on='ISO', how='left'))

# Knutson (2020) basin mapping
KNUTSON_BASIN = {
    'NATL': 'North Atlantic', 'ENP':  'Northeast Pacific',
    'WNP':  'Northwest Pacific', 'NIO':  'North Indian Ocean',
    'AUS':  'Southwest Pacific', 'SP':   'Southwest Pacific',
    'SWI':  'South Indian Ocean',
}
MANUAL_BASIN = {
    'BRB': 'North Atlantic', 'DMA': 'North Atlantic', 'ATG': 'North Atlantic',
    'VCT': 'North Atlantic', 'LCA': 'North Atlantic', 'KNA': 'North Atlantic',
    'GRD': 'North Atlantic',
    'CPV': 'North Atlantic', 'NOR': 'North Atlantic', 'FRA': 'North Atlantic',
    'TON': 'Southwest Pacific', 'WSM': 'Southwest Pacific',
    'MUS': 'South Indian Ocean', 'SYC': 'South Indian Ocean', 'COM': 'South Indian Ocean',
    'FSM': 'Northwest Pacific',
    'JPN': 'Excluded',
}
all_impact['basin'] = (
    all_impact['ISO'].map(MANUAL_BASIN)
    .fillna(all_impact['nearest_basin'].map(KNUTSON_BASIN))
    .fillna('No basin')
)
all_impact['region']      = all_impact['Region.Name'].fillna('Unknown')
all_impact['sids_status'] = np.where(all_impact['Small.Island.Developing.States..SIDS.'] == 1,
                                       'SIDS', 'Non-SIDS')

# Extratropical assignment: `extratropical_isos` from section 4, else the latitude rule.
try:
    _ = extratropical_isos
except NameError:
    _bounds = (world[['ADM0_A3', 'geometry']]
                .dropna(subset=['ADM0_A3'])
                .assign(_miny=lambda d: d.geometry.bounds['miny'],
                        _maxy=lambda d: d.geometry.bounds['maxy'])
                .groupby('ADM0_A3')
                .agg(miny=('_miny','min'), maxy=('_maxy','max')))
    extratropical_isos = set(_bounds.index[(_bounds['miny']>23.5)
                                            | (_bounds['maxy']<-23.5)]) | {'NOR'}

all_impact['is_extratropical'] = all_impact['ISO'].isin(extratropical_isos)
all_impact['region_or_extra'] = np.where(
    all_impact['is_extratropical'],
    'Extratropical',
    all_impact['region'],
)

# --- Verification print: Europe → all extratropical?  Extras from other regions?
eu_isos    = set(all_impact.loc[all_impact['region'] == 'Europe', 'ISO'])
eu_extra   = eu_isos & set(extratropical_isos)
print(f"Europe ISOs (region): {sorted(eu_isos)}  (n={len(eu_isos)})")
print(f"  ... of which extratropical: {len(eu_extra)} / {len(eu_isos)}")
extra_other = sorted(set(
    all_impact.loc[all_impact['is_extratropical'] & (all_impact['region'] != 'Europe'),
                    'ISO']
))
print(f"Extratropical ISOs originally in non-Europe regions ({len(extra_other)}): {extra_other}")

In [ ]:
tc_with_groups = TC_df[(TC_df['year'] >= 1990) & (TC_df['year'] <= 2019) & (TC_df['sovereigncountry'] == 1)].copy()
tc_with_groups = tc_with_groups.merge(
    country_master[['ISO', 'Small.Island.Developing.States..SIDS.']].drop_duplicates('ISO'),
    on='ISO', how='left'
)
tc_with_groups['sids_status'] = np.where(
    tc_with_groups['Small.Island.Developing.States..SIDS.'] == 1, 'SIDS', 'Non-SIDS'
)
# Country-years with TS-or-stronger exposure (any bin > 0), the h=5 rate-plot sample.
tc_with_groups['_bin_sum'] = (tc_with_groups['ts_v2'].fillna(0)
                       + tc_with_groups['cat1_v2'].fillna(0)
                       + tc_with_groups['cat3_v2'].fillna(0))
tc_with_groups = tc_with_groups[tc_with_groups['_bin_sum'] > 0].drop(columns=['_bin_sum'])

In [ ]:
te_groups = TC_df[(TC_df['year'] >= 1990) & (TC_df['year'] <= 2019) & (TC_df['sovereigncountry'] == 1)].copy()
te_groups = te_groups.merge(
    country_master[['ISO', 'Small.Island.Developing.States..SIDS.']].drop_duplicates('ISO'),
    on='ISO', how='left'
)
te_groups['sids_status'] = np.where(
    te_groups['Small.Island.Developing.States..SIDS.'] == 1, 'SIDS', 'Non-SIDS'
)
# Country-years with TS-or-stronger exposure (any bin > 0), the h=5 rate-plot sample.
te_groups['_bin_sum'] = (te_groups['ts_v2'].fillna(0)
                       + te_groups['cat1_v2'].fillna(0)
                       + te_groups['cat3_v2'].fillna(0))
te_groups = te_groups[te_groups['_bin_sum'] > 0].drop(columns=['_bin_sum'])

In [ ]:
a_h5 = float(coefs_debt['ts'][5])
b_h5 = float(coefs_debt['cat1'][5])
c_h5 = float(coefs_debt['cat3'][5])

rate_groups = TC_df[(TC_df['year'] >= 1990) & (TC_df['year'] <= 2019)
                     & (TC_df['sovereigncountry'] == 1)].copy()
rate_groups = rate_groups.merge(
    country_master[['ISO', 'Small.Island.Developing.States..SIDS.']].drop_duplicates('ISO'),
    on='ISO', how='left'
)
rate_groups['sids_status'] = np.where(
    rate_groups['Small.Island.Developing.States..SIDS.'] == 1, 'SIDS', 'Non-SIDS'
)
rate_groups['total_rate_h5'] = (
    rate_groups['ts_v2'].fillna(0)   * a_h5
  + rate_groups['cat1_v2'].fillna(0) * b_h5
  + rate_groups['cat3_v2'].fillna(0) * c_h5
) * 100  # convert log-rate to percent so y-axis matches the '% debt ratio' label
# Country-years with TS-or-stronger exposure (any bin > 0), as in the avg_ws and totalexposed plots.
rate_groups['_bin_sum'] = (rate_groups['ts_v2'].fillna(0)
                             + rate_groups['cat1_v2'].fillna(0)
                             + rate_groups['cat3_v2'].fillna(0))
rate_groups = rate_groups[rate_groups['_bin_sum'] > 0].drop(columns=['_bin_sum']).copy()

In [ ]:
select_countries = ['USA','PHL','BRB']

In [ ]:
# === ALTERNATIVE EXPOSURE METRIC: Debt-to-GDP counterfactual (avg_ws vs three-bin) ===
# Compare main-model three-bin LP counterfactual against a single-variable LP using
# avg_ws (annual mean wind speed across storms)

beta_avgws_debt = (coef_avgws_debt
    .query("model == 'linear' and Tshock_type == 'NATH' and term == 'avg_ws'")
    .sort_values("y_h")["estimate"]
    .values)
print(f"avg_ws (debt) coefficients: {len(beta_avgws_debt)} horizons (h=0..{len(beta_avgws_debt)-1})")


# 1980 baseline for the avg_ws comparison (kept separate from the
# 1970 PLOT_START_YEAR used by the extended series above)
PLOT_START_YEAR_ALT = 1980
# Generate yearly counterfactual rates for the three plotted countries.
results_avgws_debt = process_results(
    generate_iso_yearly_vectors_avgws(
        merged_df[merged_df["ISO"].isin(select_countries)],
        beta_avgws_debt, exposure_col="avg_ws",
        start_year=PLOT_START_YEAR_ALT, end_year=2022),
    start_year=PLOT_START_YEAR_ALT, end_year=2022)

cf_avgws_debt = calculate_country_counterfactuals(
    select_countries, observed_debt, results_avgws_debt,
    "public_pp_gdd", start_year=PLOT_START_YEAR_ALT, end_year=2022)

cf_main_debt = calculate_country_counterfactuals(
    select_countries, observed_debt, results_df_onlyTC_ext,
    "public_pp_gdd", start_year=PLOT_START_YEAR_ALT, end_year=2022)

In [ ]:
# === ALTERNATIVE EXPOSURE METRIC: GDP per capita counterfactual (avg_ws vs three-bin) ===
# Same structure as the debt comparison above, applied to log GDP per capita.

beta_avgws_gdp = (coef_avgws_gdp
    .query("model == 'linear' and Tshock_type == 'NATH' and term == 'avg_ws'")
    .sort_values("y_h")["estimate"]
    .values)
print(f"avg_ws (gdp)  coefficients: {len(beta_avgws_gdp)} horizons (h=0..{len(beta_avgws_gdp)-1})")

results_avgws_gdp = process_results(
    generate_iso_yearly_vectors_avgws(
        merged_df[merged_df["ISO"].isin(select_countries)],
        beta_avgws_gdp, exposure_col="avg_ws",
        start_year=PLOT_START_YEAR_ALT, end_year=2022),
    start_year=PLOT_START_YEAR_ALT, end_year=2022)

cf_avgws_gdp = calculate_country_counterfactuals(
    select_countries, observed, results_avgws_gdp,
    "gdp_pcap", start_year=PLOT_START_YEAR_ALT, end_year=2022)

cf_main_gdp = calculate_country_counterfactuals(
    select_countries, observed, results_df_onlyTC_gdp_ext,
    "gdp_pcap", start_year=PLOT_START_YEAR_ALT, end_year=2022)

In [ ]:
# === Per-shock-year IRFs: debt (avg_ws vs three-bin) ===
# For each plotted country, pick the top-3 shock years (largest avg_ws value
# in 1970-2014) and plot the LP-implied cumulative % change in debt ratio at
# horizons h=0..15 for that single shock under both exposure metrics.

# Pull the three-bin debt coefficients (linear, NATH).
ts_b   = (coef_debt.query("model == 'linear' and Tshock_type == 'NATH' and term == 'ts_v2'  ")
                   .sort_values("y_h")["estimate"].values)
cat1_b = (coef_debt.query("model == 'linear' and Tshock_type == 'NATH' and term == 'cat1_v2'")
                   .sort_values("y_h")["estimate"].values)
cat3_b = (coef_debt.query("model == 'linear' and Tshock_type == 'NATH' and term == 'cat3_v2'")
                   .sort_values("y_h")["estimate"].values)
H_debt = min(len(ts_b), len(beta_avgws_debt))   # common horizon range

In [ ]:
# === Per-shock-year IRFs: GDP per capita (avg_ws vs three-bin) ===
# Same per-shock-year view as the debt cell above, applied to GDP per capita
# coefficients and the GDP avg_ws coefficients.

ts_b_g   = (coef_gdp.query("model == 'linear' and Tshock_type == 'NATH' and term == 'ts_v2'  ")
                    .sort_values("y_h")["estimate"].values)
cat1_b_g = (coef_gdp.query("model == 'linear' and Tshock_type == 'NATH' and term == 'cat1_v2'")
                    .sort_values("y_h")["estimate"].values)
cat3_b_g = (coef_gdp.query("model == 'linear' and Tshock_type == 'NATH' and term == 'cat3_v2'")
                    .sort_values("y_h")["estimate"].values)
H_gdp = min(len(ts_b_g), len(beta_avgws_gdp))

In [ ]:
# Counterfactuals for all countries
all_countries_debt = calculate_country_counterfactuals(mapping_ISOs, observed_debt, results_df_onlyTC, 'public_pp_gdd')

# Map data
merged_map = prepare_map_data(world, all_countries_debt, year=2019, observed_col='public_pp_gdd', cf_col='debt_1990')
merged_map = merged_map.merge(map_key, left_on='ADMIN', right_on='ADMIN', how='left') # Add ISO column for mapping
merged_map = merged_map.merge(sovereign_mapping, left_on='ISO_A3', right_on='ISO', how='left') # for sovereign country filtering
merged_map.loc[merged_map['ISO_y'] == 'ATA', 'sovereigncountry'] = 1 # no hashmarks for Antarctica

darker = truncate_colormap(plt.get_cmap('Oranges'), 0.2, 0.8)
gradient_colors = [darker(i) for i in np.linspace(0, 1, 6)]
bins = [0, 10, 20, 30, 40, 50, 60]

map_colors = gradient_colors    # separate name from the heatmap `colors` palette
GnBu_darker2 = ListedColormap(map_colors)
norm = BoundaryNorm(bins, GnBu_darker2.N)
debt_cmap = GnBu_darker2
debt_norm = norm

In [ ]:
# Fill 'diff' values in 'merged_map' so that territories get the same color
canon = (
    merged_map.loc[merged_map["ISO_y"] == merged_map["ISO_for_mapping"], ["ISO_for_mapping", "diff"]]
    .dropna(subset=["diff"])
    .drop_duplicates(subset=["ISO_for_mapping"], keep="first")
    .set_index("ISO_for_mapping")["diff"]
)
# Assign: all rows inherit their ISO_for_mapping's canonical diff; fallback to their own diff if missing
merged_map["diff_adjusted"] = merged_map["ISO_for_mapping"].map(canon).fillna(merged_map["diff"])

In [ ]:
# 1990-start counterfactuals (TC, temp, combined) for every sovereign: debt ratio and GDP per capita.
debtratio_cf = calculate_combined_counterfactuals(
    sovereign_ISOs.tolist(), observed,
    results_df_onlyTC, results_df_onlyTEMP_delta,
    'public_pp_gdd'
)

gdp_cf = calculate_combined_counterfactuals(
    sovereign_ISOs.tolist(), observed,
    results_df_onlyTC_gdp, results_df_onlyTEMP_delta_gdp,
    'gdp_pcap'
)

In [ ]:
# cf_gdp_debtratio.csv, read by XGB_model.py (SI rating runs) and GDP_map.py (Fig. S15).
debtratio_cf = debtratio_cf.rename(columns=lambda x: x + '_debtratio' if x not in ['ISO', 'year'] else x)
gdp_cf = gdp_cf.rename(columns=lambda x: x + '_gdp' if x not in ['ISO', 'year'] else x)

combined_cf = pd.merge(debtratio_cf, gdp_cf, on=['ISO', 'year'], how='left')

combined_cf.to_csv(PATHS['cf_data'], index=False)
print(f"Saved {PATHS['cf_data']}: {combined_cf.shape}")

## 5. Figures

### Fig. 1 — `M1_rating_scatter.png`

In [ ]:
scatter = rating_data[(rating_data['year']==2019) & (rating_data['ISO'].isin(sovereign_ISOs))].copy()
scatter = scatter.merge(avg_TCexposure_by_iso, on='ISO', how='left')
scatter = scatter.merge(TEMP_2019[['ISO','meantemp_hist']], on='ISO', how='left')

scatter_nonrated = scatter[scatter['score_total'].isna()]

# Credit rating vs historical mean temperature, colored by TC exposure
plt.figure(figsize=(9, 5))

scatter_plot = scatter[scatter['score_total'] > 1.6].copy() # average rating across three agencies above CC
scatter_plot['totalexposed_capped'] = scatter_plot['totalexposed'].clip(upper=20)

# Separate zero and non-zero values
scatter_zero = scatter_plot[scatter_plot['totalexposed'] == 0]
scatter_nonzero = scatter_plot[scatter_plot['totalexposed'] > 0]

# Plot zero values first (in the back) with lighter grey and more transparency
plt.scatter(
    x=scatter_zero['meantemp_hist'],
    y=scatter_zero['score_total'],
    color='#C0C0C0',  # very light grey
    edgecolor='none',
    s=50,
    alpha=0.3,  # More transparent
    zorder=1  # Lower z-order to be in back
)

# Create custom colormap for non-zero values:
colors_list = sns.color_palette('Blues', n_colors=100)[20:]  # Skip very light blues
custom_cmap = mcolors.LinearSegmentedColormap.from_list('custom_blues', colors_list)

# Plot non-zero values on top
sns.scatterplot(
    x='meantemp_hist', 
    y='score_total', 
    data=scatter_nonzero, 
    hue='totalexposed_capped', 
    palette=custom_cmap, 
    edgecolor='none',
    s=50,
    legend=False,
    zorder=2  # Higher z-order to be on top
)

plt.title('')
plt.axhline(y=12, color='grey', linestyle='--', alpha=0.7)

sns.regplot(
    x='meantemp_hist', 
    y='score_total', 
    data=scatter, 
    scatter=False, 
    color='grey', 
    line_kws={'label': 'Linear Fit', 'alpha': 0.2, 'linestyle': 'dotted'},
    ci=None
)

# Add ISO labels at each point
label_ISOs = ['USA','BHS', 'BRB' ,'PHL','JAM', 'VEN',
              'BLZ','FJI','SUR','CPV','MNG','FIN','MEX','CRI', 'GBR','GEO','BOL','CZE', 'SGP', 'KWT', 'SAU', 'QAT', 'ARE', 'MNG', 'NOR', 'FIN', 'KOR', 'JPN', 'ISL', 'RUS', 'ARM', 'LBN', 'ARG', 'KEN','CUB', 'GTM', 'MDG']

# Add text labels only for selected ISOs with x_jitter
for i, row in scatter.iterrows():
    if row['ISO'] in label_ISOs:
        x_jitter = 0.3
        y_jitter = 0.03
        plt.text(
            row['meantemp_hist'] + x_jitter, 
            row['score_total'] + y_jitter, 
            row['ISO'], 
            fontsize=8, 
            alpha=0.7
        )

ax = plt.gca()

# Create colorbar with capped values (using non-zero range)
sm = plt.cm.ScalarMappable(cmap=custom_cmap, norm=plt.Normalize(vmin=0, vmax=20))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, orientation='vertical', shrink=0.4, extend='max', pad=0.15)
cbar.set_label('Average tropical cyclone exposure (% area)', size=8)
cbar.set_ticks([0, 5, 10, 15, 20])
cbar.set_ticklabels(['0', '5', '10', '15', '≥20'])

# Add "Investment grade" and "Speculative grade" labels in the space between plot and colorbar
# Position is in axes coordinates (right edge of plot area)
ax.text(1.02, 0.75, 'Investment\ngrade', transform=ax.transAxes, fontsize=10, 
        va='center', ha='left', color='black')
ax.text(1.02, 0.35, 'Speculative\ngrade', transform=ax.transAxes, fontsize=10, 
        va='center', ha='left', color='black')
ax.text(1.02, -0.027, 'non-rated', transform=ax.transAxes, fontsize=10, 
        va='center', ha='left', color='black')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.yticks(
    ticks=[2, 12, 21], 
    labels=['CC', 'BBB-', 'AAA']
)
plt.xlim(-2, 29)
plt.ylim(1.5, 22)

# Add rug plot below the x-axis as circles matching scatter plot style
# Separate zero and non-zero exposure values for non-rated countries
scatter_nonrated_zero = scatter_nonrated[scatter_nonrated['totalexposed'] == 0]
scatter_nonrated_nonzero = scatter_nonrated[scatter_nonrated['totalexposed'] > 0].copy()
scatter_nonrated_nonzero['totalexposed_capped'] = scatter_nonrated_nonzero['totalexposed'].clip(upper=20)

# Create normalizer for coloring
scatter_norm = plt.Normalize(vmin=0, vmax=20)

# Plot zero exposure non-rated countries (grey circles)
ax.scatter(
    x=scatter_nonrated_zero['meantemp_hist'],
    y=[0.3] * len(scatter_nonrated_zero),  # Position below the plot area
    color='#C0C0C0',
    edgecolor='none',
    s=50,
    alpha=0.3,
    clip_on=False,
    zorder=3
)

# Plot non-zero exposure non-rated countries (colored circles)
ax.scatter(
    x=scatter_nonrated_nonzero['meantemp_hist'],
    y=[0.9] * len(scatter_nonrated_nonzero),  # Position below the plot area
    c=scatter_nonrated_nonzero['totalexposed_capped'],
    cmap=custom_cmap,
    norm=scatter_norm,
    edgecolor='none',
    s=40,
    alpha=0.9,
    clip_on=False,
    zorder=3
)

plt.sca(ax)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Increase padding between x-axis and tick labels
ax.tick_params(axis='x', pad=25, length=0)

plt.xlabel('Mean Temperature (°C)', fontsize=11)
plt.ylabel('Credit Rating', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.savefig("./4_figures/main/M1_scatter/M1_rating_scatter.png", dpi=300, bbox_inches='tight')
plt.show()

### Fig. 2 panel a — TC impact on debt-to-GDP

In [ ]:
# Debt Ratio IRF PLOTTING 
fig, axes = plot_TC_debt(
    df=TC_main,
    figsize=(20, 2),
    outcome='public_pp_GDD',
    specification='1980onwards_gl2_TC10_T02' # MAIN MODEL (117 countries w/ >40 yrs data)
)

fig.text(
    0.07, # horizontal
    1.1, # vertical
    "a",
    ha="left", va="top",
    fontsize=16, fontweight="bold", color="black"
)
plt.suptitle("Tropical Cyclone Impact on Debt-to-GDP", fontsize=15, fontweight='bold',
             x=0.19,
             y=1.3)

plt.show()

### Fig. 2 panel b — TC impact on debt ratio (contour)

In [ ]:
# DEBT RATIO HEATMAPS - using plot_heatmap helper
fig, axes = plt.subplots(1, 3, figsize=(18.5, 3))

pivot_data_ts = grid_debt.pivot(index="area", columns="years", values="value_ts")
smoothed_values_ts = gaussian_filter(pivot_data_ts.values, sigma=sigma_value)
smoothed_data_ts = pd.DataFrame(smoothed_values_ts, index=pivot_data_ts.index, columns=pivot_data_ts.columns)

plot_heatmap(axes[0], smoothed_data_ts, cmap=sns.light_palette(colors[0], as_cmap=True), tick_positions=tick_positions, tick_labels=tick_labels)
# Add labels to the contour lines for TS
data_values = smoothed_data_ts.values
X, Y = np.meshgrid(np.arange(data_values.shape[1]), np.arange(data_values.shape[0]))
levels = [5, 10, 15]
CS = axes[0].contour(X, Y, data_values, levels=levels, colors='w', linewidths=0.7)
fmt = lambda x: f"{x:.0f}%"  # whole-number percent labels
axes[0].clabel(CS, inline=True, fontsize=8, fmt=fmt)
axes[0].set_ylabel("% area exposed", fontsize=12)

pivot_data_cat1 = grid_debt.pivot(index="area", columns="years", values="value_cat1")
smoothed_values_cat1 = gaussian_filter(pivot_data_cat1.values, sigma=sigma_value)
smoothed_data_cat1 = pd.DataFrame(smoothed_values_cat1, index=pivot_data_cat1.index, columns=pivot_data_cat1.columns)

plot_heatmap(axes[1], smoothed_data_cat1, cmap=sns.light_palette(colors[1], as_cmap=True), tick_positions=tick_positions, tick_labels=tick_labels)
# Add labels to the contour lines for Cat1
data_values_cat1 = smoothed_data_cat1.values
X, Y = np.meshgrid(np.arange(data_values_cat1.shape[1]), np.arange(data_values_cat1.shape[0]))
levels_cat1 = [5, 10, 15, 20]
CS_cat1 = axes[1].contour(X, Y, data_values_cat1, levels=levels_cat1, colors='w', linewidths=0.7)
axes[1].clabel(CS_cat1, inline=True, fontsize=10, fmt=fmt)
axes[1].set_ylabel("")

pivot_data_cat3 = grid_debt.pivot(index="area", columns="years", values="value_cat3")
smoothed_values_cat3 = gaussian_filter(pivot_data_cat3.values, sigma=sigma_value)
smoothed_data_cat3 = pd.DataFrame(smoothed_values_cat3, index=pivot_data_cat3.index, columns=pivot_data_cat3.columns)

plot_heatmap(axes[2], smoothed_data_cat3, cmap=sns.light_palette(colors[2], as_cmap=True), tick_positions=tick_positions, tick_labels=tick_labels)
# Add labels to the contour lines for Cat3
data_values_cat3 = smoothed_data_cat3.values
X, Y = np.meshgrid(np.arange(data_values_cat3.shape[1]), np.arange(data_values_cat3.shape[0]))
levels_cat3 = [5, 15, 30, 40]
CS_cat3 = axes[2].contour(X, Y, data_values_cat3, levels=levels_cat3, colors='w', linewidths=0.7)
axes[2].clabel(CS_cat3, inline=True, fontsize=10, fmt=fmt)
axes[2].set_ylabel("")

# Only show 0, 10, 20 for tick labels
cbar = axes[2].collections[0].colorbar
ticks = cbar.get_ticks()
filtered_ticks = [tick for tick in ticks if tick in [0, 10, 20, 30, 40]]
cbar.set_ticks(filtered_ticks)

sns.rugplot(ax=axes[0], height=0.03, color=colors[0], alpha=0.2, y=ts_dist, clip_on=True, linewidth=1.5)
sns.rugplot(ax=axes[1], height=0.03, color=colors[1], alpha=0.7, y=cat1_dist, clip_on=True, linewidth=1.5)
sns.rugplot(ax=axes[2], height=0.03, color=colors[2], alpha=0.7, y=cat3_dist, clip_on=True, linewidth=1.5)

fig.text(
    -0.01, # horizontal
    1, # vertical
    "b",
    ha="left", va="top",
    fontsize=18, fontweight="bold", color="black"
)

plt.tight_layout()
plt.savefig("./4_figures/main/M2_IRF/TC_debt_IRF_contour.png", dpi=300, bbox_inches='tight')
plt.show()

### Fig. 2 panel c — TC impact on GDP per capita

In [ ]:
fig, axes = plot_TC_gdp(
    df=TC_main,
    subset='full',
    outcome='gdp_pcap_wdi_constant2015_USD',
    specification='1970onwards_gl2_TC10_T02', # MAIN MODEL
    model='linear',
    figsize=(20, 2),
    ylim=(-0.005, 0.0005),
    ylabel='Δ GDP pcap / \n % area exposed',
    pct_format=True
)
fig.text(
    0.07, # horizontal
    1.1, # vertical
    "c",
    ha="left", va="top",
    fontsize=18, fontweight="bold", color="black"
)

plt.suptitle("Tropical Cyclone Impact on GDP per capita", fontsize=15, fontweight='bold',
             x=0.19,
             y=1.3)

plt.show()

### Fig. 2 panel d — TC impact on GDP per capita (contour)

In [ ]:
# GDP PER CAPITA HEATMAPS - using create_heatmap_grid and plot_heatmap helpers

fig, axes = plt.subplots(1, 3, figsize=(18.5, 3))

years_list = list(grid_gdp.pivot(index="area", columns="years", values="value_ts").columns)
desired_ticks = [0, 5, 10, 15]
tick_positions = [i for i, year in enumerate(years_list) if year in desired_ticks]
tick_labels = [int(year) for year in years_list if year in desired_ticks]

colors = ["#B3D9F2", "#4682B4", "#1C4E80"]  # blues 2

pivot_data_ts = grid_gdp.pivot(index="area", columns="years", values="value_ts")
smoothed_values_ts = gaussian_filter(pivot_data_ts.values, sigma=sigma_value)
smoothed_data_ts = pd.DataFrame(smoothed_values_ts, index=pivot_data_ts.index, columns=pivot_data_ts.columns)

plot_heatmap(axes[0], smoothed_data_ts, cmap=sns.light_palette(colors[0], as_cmap=True), tick_positions=tick_positions, tick_labels=tick_labels)
# Add labels to the contour lines for TS
data_values = smoothed_data_ts.values
X, Y = np.meshgrid(np.arange(data_values.shape[1]), np.arange(data_values.shape[0]))
levels = [0, 0.5, 1, 1.5, 2, 3]
CS = axes[0].contour(X, Y, data_values, levels=levels, colors='w', linewidths=0.7)
fmt = lambda x: f"{x:.1f}%"  # one-decimal percent labels
axes[0].clabel(CS, inline=True, fontsize=8, fmt=fmt)
axes[0].set_ylabel("% area exposed", fontsize=12)

pivot_data_cat1 = grid_gdp.pivot(index="area", columns="years", values="value_cat1")
smoothed_values_cat1 = gaussian_filter(pivot_data_cat1.values, sigma=sigma_value)
smoothed_data_cat1 = pd.DataFrame(smoothed_values_cat1, index=pivot_data_cat1.index, columns=pivot_data_cat1.columns)

plot_heatmap(axes[1], smoothed_data_cat1, cmap=sns.light_palette(colors[1], as_cmap=True), tick_positions=tick_positions, tick_labels=tick_labels)
# Add labels to the contour lines for Cat1
data_values_cat1 = smoothed_data_cat1.values
X, Y = np.meshgrid(np.arange(data_values_cat1.shape[1]), np.arange(data_values_cat1.shape[0]))
levels_cat1 = [0, 0.01, 0.25, 0.5, 1]
CS_cat1 = axes[1].contour(X, Y, data_values_cat1, levels=levels_cat1, colors='w', linewidths=0.7)
axes[1].clabel(CS_cat1, inline=True, fontsize=8, fmt=fmt)
axes[1].set_ylabel("", fontsize=12)

pivot_data_cat3 = grid_gdp.pivot(index="area", columns="years", values="value_cat3")
smoothed_values_cat3 = gaussian_filter(pivot_data_cat3.values, sigma=sigma_value)
smoothed_data_cat3 = pd.DataFrame(smoothed_values_cat3, index=pivot_data_cat3.index, columns=pivot_data_cat3.columns)

plot_heatmap(axes[2], smoothed_data_cat3, cmap=sns.light_palette(colors[2], as_cmap=True), tick_positions=tick_positions, tick_labels=tick_labels)
# Add labels to the contour lines for Cat3
data_values_cat3 = smoothed_data_cat3.values
X, Y = np.meshgrid(np.arange(data_values_cat3.shape[1]), np.arange(data_values_cat3.shape[0]))
levels_cat3 = [1, 2.5, 5, 10, 20]
CS_cat3 = axes[2].contour(X, Y, data_values_cat3, levels=levels_cat3, colors='w', linewidths=0.7)
axes[2].clabel(CS_cat3, inline=True, fontsize=8, fmt=fmt)
axes[2].set_ylabel("", fontsize=12)

# Only show 0, 10, 20 for tick labels
cbar = axes[2].collections[0].colorbar
ticks = cbar.get_ticks()
filtered_ticks = [tick for tick in ticks if tick in [0, 0.01, 0.25, 0.5, 1, 1.5, 2, 3, 4, 6]]
cbar.set_ticks(filtered_ticks)

sns.rugplot(ax=axes[0], height=0.03, color=colors[0], alpha=0.2, y=ts_dist, clip_on=True, linewidth=1.4)
sns.rugplot(ax=axes[1], height=0.03, color=colors[1], alpha=0.7, y=cat1_dist, clip_on=True, linewidth=1.5)
sns.rugplot(ax=axes[2], height=0.03, color=colors[2], alpha=0.7, y=cat3_dist, clip_on=True, linewidth=1.5)

fig.text(
    -0.01, # horizontal
    1, # vertical
    "d",
    ha="left", va="top",
    fontsize=18, fontweight="bold", color="black"
)

plt.tight_layout()
plt.savefig("./4_figures/main/M2_IRF/TC_GDP_IRF_contour.png", dpi=300, bbox_inches='tight')
plt.show()

### Fig. 3 panel a — debt-ratio counterfactual time series

In [ ]:
df = combined_TC_ext
selectoutcome = 'public_pp_gdd'
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 2.5))
titles = ['United States', 'The Philippines', 'Barbados']

for i, iso in enumerate(select_countries):
    plot_cf2 = df[(df['ISO'] == iso) & (df['year'] <= 2019)]
    ax = axes[i]

    cmap     = plt.get_cmap('Oranges')
    startyear = PLOT_START_YEAR
    years_iter = list(range(startyear, 2021, 1))
    n_orange = max(2021 - GREY_END_YEAR, 1)   # color steps over the orange-shaded years

    for year in years_iter:
        col = f'debt_{year}'
        if col not in plot_cf2.columns:
            continue
        ax.plot(plot_cf2['year'], plot_cf2[col], linestyle='--', color="white",
                linewidth=0.1, alpha=0.3)

        # Pre-1980 storm years stay lightgrey (same shade as the area below the
        # earliest counterfactual); 1980+ uses the orange gradient.
        if year < GREY_END_YEAR:
            fill_color, fill_alpha = "lightgrey", 0.7
        else:
            j_orange = year - GREY_END_YEAR
            fill_color = cmap(0.3 + 0.6 * j_orange / max(n_orange - 1, 1))
            fill_alpha = 0.3

        if year == startyear:
            ax.fill_between(plot_cf2['year'], 0, plot_cf2[col],
                            color=fill_color, alpha=fill_alpha)
        else:
            prev = f'debt_{year-1}'
            if prev in plot_cf2.columns:
                ax.fill_between(plot_cf2['year'], plot_cf2[prev], plot_cf2[col],
                                color=fill_color, alpha=fill_alpha)

    # Vertical reference line at the grey/orange boundary
    ax.axvline(x=GREY_END_YEAR, color='grey', linestyle=':', linewidth=1, alpha=0.7)

    ax.plot(plot_cf2['year'], plot_cf2[selectoutcome], color='black',
            linewidth=2, alpha=0.7, label='observed')
    ax.plot(plot_cf2['year'], plot_cf2[f'debt_{startyear}'],
            linestyle='--', color="black", linewidth=1.5,
            label=f'counterfactual')
    if i == 0:
        ax.legend(fontsize='small', loc='lower left')
    if i < len(titles):
        ax.set_title(titles[i], fontsize=12, fontweight='bold', loc='left')
    if i == 0:
        ax.set_ylabel('Debt Ratio (%)', fontsize=9)
    ax.set_xticks(ticks=range(startyear, 2021, 10))
    ax.set_xticklabels(range(startyear, 2021, 10), fontsize=10)
    ax.grid(False)
    ax.set_ylim(0, None)
    ax.set_xlim(startyear, 2021)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color('black')
    ax.spines[['left', 'bottom']].set_linewidth(0.5)

plt.tight_layout()
fig.text(x=0, y=0.95, s="a", ha="left", va="top",
         fontsize=20, fontweight="bold", color="black")
plt.subplots_adjust(top=0.85)
plt.suptitle(f'Debt-to-GDP time series',
             fontsize=16, fontweight='bold', x=0.1, y=1.1)
plt.savefig(f"./4_figures/main/M3_counterfactuals/debt_ratio_time_series_ext{PLOT_START_YEAR}.png",
            dpi=300, bbox_inches='tight')
plt.show()

### Fig. 3 panel b — TC exposure time series

In [ ]:
# --- TC EXPOSURE time series (panel b: aligned width with debt CF panel above) ---
# Three lines per panel (TS / Cat1-3 / Cat >3); figsize matches the debt panels above.
exposure_cols   = ['ts_v2', 'cat1_v2', 'cat3_v2']
exposure_labels = ['TS', 'Cat1-3', 'Cat >3']
tc_colors       = ['#B5A8D5', '#7A73D1', '#4D55CC']

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 1.8))
for i, iso in enumerate(select_countries):
    ax = axes[i]
    sub = (merged_df[(merged_df['ISO'] == iso) &
                     (merged_df['year'] >= PLOT_START_YEAR) &
                     (merged_df['year'] <= 2019)]
            .sort_values('year'))

    for col, lbl, c in zip(exposure_cols, exposure_labels, tc_colors):
        ax.plot(sub['year'], sub[col],
                color=c, linewidth=1.2, label=lbl)

    if i == 0:
        ax.legend(fontsize=8, frameon=False, loc='upper left',
                  handlelength=1.5, handletextpad=0.5)
        ax.set_ylabel('% area exposed', fontsize=9)

    ax.set_xticks(range(PLOT_START_YEAR, 2021, 10))
    ax.set_xticklabels(range(PLOT_START_YEAR, 2021, 10), fontsize=10)
    ax.set_xlim(PLOT_START_YEAR, 2021)
    ax.set_ylim(0, None)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color('black')
    ax.spines[['left', 'bottom']].set_linewidth(0.5)
    ax.grid(False)

plt.tight_layout()
fig.text(x=-0.01, y=0.98, s='b', ha='left', va='top',
         fontsize=19, fontweight='bold', color='black')
plt.savefig(f"./4_figures/main/M3_counterfactuals/tc_exposure_time_series_ext{PLOT_START_YEAR}.png",
            dpi=300, bbox_inches='tight')
plt.show()

### Fig. 3 panel c — GDP per capita counterfactual time series

In [ ]:
df = gdp_combined_cf_ext
selectoutcome = 'gdp_pcap'

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 2.5))
titles = ['United States', 'The Philippines', 'Barbados']

cf_temp_col     = f'{PLOT_START_YEAR}_cf_temp'
cf_combined_col = f'{PLOT_START_YEAR}_cf_combined'

for i, iso in enumerate(select_countries):
    plot_cf2 = df[(df['ISO'] == iso) & (df['year'] <= 2019)].sort_values('year')
    ax = axes[i]

    observed_line, = ax.plot(plot_cf2['year'], plot_cf2[selectoutcome],
                              color='black', linewidth=2, alpha=0.7, label='observed')
    ax.fill_between(plot_cf2['year'], 0, plot_cf2[selectoutcome],
                    color='lightgrey', alpha=0.5)
    ax.fill_between(plot_cf2['year'], plot_cf2[selectoutcome], plot_cf2[cf_temp_col],
                    color='salmon', alpha=0.25)
    ax.plot(plot_cf2['year'], plot_cf2[cf_temp_col],
            linestyle='--', color='salmon', linewidth=1.5)
    ax.fill_between(plot_cf2['year'], plot_cf2[cf_temp_col], plot_cf2[cf_combined_col],
                    color='#4169E1', alpha=0.25)
    counterfactual_line, = ax.plot(plot_cf2['year'], plot_cf2[cf_combined_col],
                                    linestyle='--', color='black', linewidth=1.5,
                                    label='counterfactual')
    if i == 0:
        legend_handles = [
            observed_line,
            counterfactual_line,
            Rectangle((0, 0), 1, 1, facecolor='salmon', alpha=0.25, edgecolor='none'),
            Rectangle((0, 0), 1, 1, facecolor='#4169E1', alpha=0.25, edgecolor='none')
        ]
        legend_labels = ['observed', 'counterfactual', 'Temp impact', 'TC impact']
        ax.legend(legend_handles, legend_labels, fontsize='small', loc='lower left', ncol=2)

    if i < len(titles):
        ax.set_title(titles[i], fontsize=12, fontweight='bold', loc='left')
    if i == 0:
        ax.set_ylabel('GDP per capita ($2015)', fontsize=9)

    # Vertical reference line at GREY_END_YEAR (matches the debt-ratio plot above)
    ax.axvline(x=GREY_END_YEAR, color='grey', linestyle=':', linewidth=1, alpha=0.7)

    ax.set_xticks(ticks=range(PLOT_START_YEAR, 2021, 10))
    ax.set_xticklabels(range(PLOT_START_YEAR, 2021, 10), fontsize=10)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: '0' if x == 0 else f'{int(x/1000)}k'))
    ax.grid(False)
    ax.set_ylim(0, None)
    ax.set_xlim(PLOT_START_YEAR, 2021)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color('black')
    ax.spines[['left', 'bottom']].set_linewidth(0.5)

plt.tight_layout()

fig.text(
    x=0,
    y=0.95,
    s="c",
    ha="left",
    va="top",
    fontsize=20,
    fontweight="bold",
    color="black"
)

plt.subplots_adjust(top=0.85)
plt.suptitle('GDP per capita time series', fontsize=16, fontweight='bold', x=0.1, y=1.1)
plt.savefig("./4_figures/main/M3_counterfactuals/gdp_time_series.png", dpi=300, bbox_inches='tight')
plt.show()

### Fig. 4 panel a — global map with TC tracks

In [ ]:
# Global map
output_dir = "./4_figures/main/M4_map/"

fig = plt.figure(figsize=(12, 5))
ax = plt.axes(projection=ccrs.Robinson())
ax.set_facecolor((0.91, 0.98, 1, 0.8))

# Crop Antarctica: limit to latitudes -70 to 90
robinson = ccrs.Robinson()
_, y_min = robinson.transform_point(0, -80, ccrs.PlateCarree())
_, y_max = robinson.transform_point(0, 80, ccrs.PlateCarree())
ax.set_ylim(y_min, y_max)

sm = plt.cm.ScalarMappable(cmap=GnBu_darker2, norm=norm)
sm.set_array([])  # Empty array to avoid warnings
cbar = fig.colorbar(sm, ax=ax, orientation='horizontal', fraction=0.02, pad=0.04)
cbar.set_label('debt ratio impact (%)', labelpad=-40, y=1.05, fontsize=10)
# Place the colorbar inside the map, lower-left quadrant
cbar.ax.set_position([0.4, 0.25, 0.2, 0.02])
# Add a white box background to the legend
cbar.ax.set_facecolor('white')

merged_map.plot(
    column='diff_adjusted',
    cmap=GnBu_darker2,
    norm=norm,
    ax=ax,
    edgecolor='grey',
    linewidth=0.5,
    alpha=0.8,
    transform=ccrs.PlateCarree(),
    missing_kwds={
        "color": "lightgrey",
        "label": "No data",
        "alpha": 0.3
    }
)

# Overlay: hatch non-sovereign polygons
merged_map.loc[merged_map["sovereigncountry"] == 0].plot(
    ax=ax,
    facecolor="none",        # crucial: don't overwrite colors
    edgecolor="grey",
    linewidth=0.5,
    hatch="///",
    transform=ccrs.PlateCarree()
)
# Plot tracks by storm ID (blue overlay on panel a; empty frame -> no-op
# when the external IBTrACS file is not present, see 2_code/data_prep/DATA.md)
for sid, group in filtered_ibtracs.groupby('SID'):
    ax.plot(
        group['LON'],
        group['LAT'],
        marker='o',
        markersize=0.1,
        linestyle='-',
        linewidth=0.1,
        alpha=0.2,
        label=None,
        color='#3D90D7',
        transform=ccrs.PlateCarree()
    )

box_label_color = 'black'
# Latin America
lat_min = 10
lat_max = 30
lon_min = -90
lon_max = -58

box1 = Rectangle(
    (lon_min, lat_min),
    lon_max - lon_min,
    lat_max - lat_min,
    linewidth=1,
    edgecolor='black',
    linestyle='--',
    facecolor='none',
    transform=ccrs.PlateCarree()
)
ax.add_patch(box1)
ax.text(lon_min + 3, lat_max, "b", color=box_label_color, fontsize=15, fontweight='bold', ha='right', va='bottom', transform=ccrs.PlateCarree())

# CARIBBEAN
lat_min = 10
lat_max = 18.5
lon_min = -63.5
lon_max = -59.1
box2 = Rectangle(
    (lon_min, lat_min),
    lon_max - lon_min,
    lat_max - lat_min,
    linewidth=1,
    edgecolor='black',
    linestyle='--',
    facecolor='none',
    transform=ccrs.PlateCarree()
)
ax.add_patch(box2)
ax.text(lon_min + 3, lat_max, "c", color="black", fontsize=15, fontweight="bold", ha="right", va="bottom", transform=ccrs.PlateCarree())

# SOUTHEAST ASIA
lat_min = -12
lat_max = 27
lon_min = 85
lon_max = 135
box3 = Rectangle(
    (lon_min, lat_min),
    lon_max - lon_min,
    lat_max - lat_min,
    linewidth=1,
    edgecolor='black',
    linestyle='--',
    facecolor='none',
    transform=ccrs.PlateCarree()
)
ax.add_patch(box3)
ax.text(lon_min + 3, lat_max, "d", color="black", fontsize=15, fontweight="bold", ha="right", va="bottom", transform=ccrs.PlateCarree())

# PACIFIC ISLANDS
lat_min = -25
lat_max = -5
lon_min = 155
lon_max = 180
box4 = Rectangle(
    (lon_min, lat_min),
    lon_max - lon_min,
    lat_max - lat_min,
    linewidth=1,
    edgecolor='black',
    linestyle='--',
    facecolor='none',
    transform=ccrs.PlateCarree()
)
ax.add_patch(box4)
ax.text(lon_min + 3, lat_max, "e", color="black", fontsize=15, fontweight="bold", ha="right", va="bottom", transform=ccrs.PlateCarree())

ax.axhline(y=0, color='grey', linestyle='--', linewidth=0.6, alpha=0.5)
fig.text(
    0.2, # horizontal
    0.87, # vertical
    "a",
    ha="left", va="top",
    fontsize=25, fontweight="bold", color="black"
)
plt.savefig(os.path.join(output_dir, 'panel_A.png'), dpi=300, bbox_inches='tight')
plt.show()

### Fig. 4 panels b and c — Caribbean / Lesser Antilles

In [ ]:
# Regional maps (panels b-e)
import matplotlib as mpl
mpl.rcParams["hatch.linewidth"] = 0.4

merged = merged_map

caribbean_only = merged[merged['SUBREGION'].isin(['Caribbean'])]
caribbean_labels = caribbean_only[~caribbean_only['GEOUNIT'].isin([
    'The Bahamas', 'Jamaica', 'Cuba', 'Haiti', 'Dominican Republic', 'Aruba', 'Trinidad and Tobago'
])]['GEOUNIT'].unique()

# Panel b: Latin America + Caribbean
merged1 = merged[merged['SUBREGION'].isin(['Caribbean', 'Central America', 'Northern America', 'South America'])]

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
fig.patch.set_facecolor('none')
ax.set_facecolor((0.91, 0.98, 1, 1))

merged1.plot(
    column='diff_adjusted',
    cmap=GnBu_darker2,
    norm=norm,
    edgecolor='black',
    linewidth=0.4,
    legend=False,
    ax=ax,
    missing_kwds={"color": "lightgrey", "alpha": 0.5, "label": "No data"},
    vmax=40
)

# Overlay: hatch territories without its own debt data
merged1.loc[merged1["sovereigncountry"] == 0].plot(
    ax=ax,
    facecolor="none",        # crucial: don't overwrite colors
    edgecolor="grey",
    linewidth=0.1,
    hatch='//////'
)

lat_min = 10.5
lat_max = 28
lon_min = -90
lon_max = -58

ax.set_ylim(lat_min, lat_max)
ax.set_xlim(lon_min, lon_max)
ax.set_xticks([])
ax.set_yticks([])

# Caribbean box
box_lat_min = 10.5
box_lat_max = 18.5
box_lon_min = -63.5
box_lon_max = -59.1
ax.plot(
    [box_lon_min, box_lon_max, box_lon_max, box_lon_min, box_lon_min],
    [box_lat_min, box_lat_min, box_lat_max, box_lat_max, box_lat_min],
    color="black",
    linestyle="--",
    linewidth=1,
    alpha=0.8
)

label_fontsize = 7
for x, y, label in zip(merged1.geometry.centroid.x, merged1.geometry.centroid.y, merged1['GEOUNIT']):
    if lon_min <= x <= lon_max and lat_min <= y <= lat_max and label not in ['Barbados', 'Trinidad and Tobago'] + list(caribbean_labels):
        if label == "The Bahamas":
            ax.text(x + 3, y + 2, label, fontsize=label_fontsize, ha='center', color='black')
        elif label == "Dominican Republic":
            ax.text(x + 3, y + 1, "Dominican \nRepublic", fontsize=label_fontsize, ha='center', color='black')
        elif label == "Haiti":
            ax.text(x + 0, y - 2, label, fontsize=label_fontsize, ha='center', color='black')
        elif label == "Jamaica":
            ax.text(x + 0, y - 1, label, fontsize=label_fontsize, ha='center', color='black')
        elif label == "Belize":
            ax.text(x + 2.2, y + 0, label, fontsize=label_fontsize, ha='center', color='black')
        elif label == "El Salvador":
            ax.text(x + 1.2, y - 1, label, fontsize=label_fontsize, ha='center', color='black')
        elif label == "Nicaragua":
            ax.text(x + 1.5, y - 1, label, fontsize=label_fontsize, ha='center', color='black')
        else:
            ax.text(x, y, label, fontsize=label_fontsize, ha='center', color='black')

ax.text(box_lon_max - 0.4, box_lat_max - 1.6, "c", color="black", fontsize=16, fontweight="bold", ha="right", va="bottom")
ax.text(lon_max - 0.8, lat_max - 2, "b", color="black", fontsize=16, fontweight="bold", ha="right", va="bottom")

plt.savefig(os.path.join(output_dir, 'panel_b.png'), dpi=300, bbox_inches='tight')
plt.show()

### Fig. 4 panels c, d, e — zoom maps

In [ ]:
# Panel c: Caribbean zoom
merged1 = merged[merged['REGION_WB'].isin(['Latin America & Caribbean', 'North America'])]
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
fig.patch.set_facecolor('none')
ax.set_facecolor((0.91, 0.98, 1, 1))

merged1.plot(
    column='diff_adjusted',
    cmap=GnBu_darker2,
    norm=norm,
    edgecolor='black',
    linewidth=0.3,
    legend=False,
    ax=ax,
    alpha=1,
    missing_kwds={"color": "lightgrey", "alpha": 0.5, "label": "No data"},
    vmax=40
)

# Overlay: hatch territories without its own debt data
merged1.loc[merged1["sovereigncountry"] == 0].plot(
    ax=ax,
    facecolor="none",        # crucial: don't overwrite colors
    edgecolor="grey",
    linewidth=0.05,
    hatch="/////"
)

lat_min = 10.5
lat_max = 18.5
lon_min = -63.5
lon_max = -59.1

ax.set_ylim(lat_min, lat_max)
ax.set_xlim(lon_min, lon_max)
ax.set_xticks([])
ax.set_yticks([])

# Dissolve so each GEOUNIT is a single geometry, then label via representative points
merged1_lab = merged1.dissolve(by="GEOUNIT", as_index=False)
pts = merged1_lab.geometry.representative_point()

label_fontsize = 7
for x, y, label in zip(pts.x, pts.y, merged1_lab["GEOUNIT"]):

    if label == "Saint Kitts and Nevis":
        ax.text(x + 0.0, y - 0.7, "Saint Kitts\nand Nevis",
                fontsize=label_fontsize, ha="center", color="black")

    elif label == "Saint Vincent and the Grenadines":
        ax.text(x - 0.9, y - 0.4, "Saint Vincent\nand the Grenadines",
                fontsize=label_fontsize, ha="center", color="black")

    elif label == "Grenada":
        ax.text(x - 0.6, y + 0.0, "Grenada",
                fontsize=label_fontsize, ha="center", color="black")

    elif label == "Barbados":
        ax.text(x - 0.3, y - 0.4, "Barbados",
                fontsize=label_fontsize, ha="center", color="black")

    elif label == "Dominica":
        ax.text(x - 0.7, y - 0.1, "Dominica",
                fontsize=label_fontsize, ha="center", color="black")

    elif label == "Saint Lucia":
        ax.text(x - 0.8, y - 0.1, "Saint Lucia",
                fontsize=label_fontsize, ha="center", color="black")

    elif label == "Barbuda":
        ax.text(x + 1.0, y - 0.3, "Antigua\nand Barbuda",
                fontsize=label_fontsize, ha="center", color="black")

    elif label == "Trinidad and Tobago":
        ax.text(x + 0.0, y + 0.6, "Trinidad\nand Tobago",
                fontsize=label_fontsize, ha="center", color="black")


ax.text(lon_max - 0.2, lat_max - 0.8, "c", color="black", fontsize=17.5, fontweight="bold", ha="right", va="bottom")

plt.savefig(os.path.join(output_dir, 'panel_c.png'), dpi=300, bbox_inches='tight')
plt.show()

# Panel d: SOUTHEAST ASIA
merged1 = merged[merged['SUBREGION'].isin(['South-Eastern Asia', 'Southern Asia', "Eastern Asia"])]

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
fig.patch.set_facecolor('none')
ax.set_facecolor((0.91, 0.98, 1, 1))

merged1.plot(
    column='diff_adjusted',
    cmap=GnBu_darker2,
    norm=norm,
    edgecolor='black',
    linewidth=0.3,
    legend=False,
    ax=ax,
    missing_kwds={"color": "lightgrey", "alpha": 0.5, "label": "No data"},
    vmax=40
)

# Overlay: hatch territories without its own debt data
merged1.loc[merged1["sovereigncountry"] == 0].plot(
    ax=ax,
    facecolor="none",        # crucial: don't overwrite colors
    edgecolor="grey",
    linewidth=0.05,
    hatch="/////",
    aspect=None              # keep the base layer's aspect
)

lat_min = -11.5
lat_max = 26.5
lon_min = 90
lon_max = 141

label_fontsize = 6
for x, y, label in zip(merged1.geometry.centroid.x, merged1.geometry.centroid.y, merged1['GEOUNIT']):
    if lon_min <= x <= lon_max and lat_min <= y <= lat_max and label not in [
        "Scarborough Reef", "Spratly Islands", "Christmas Island", "Taiwan",
        "Hong Kong S.A.R.", "Macao S.A.R", "Guam", "Northern Mariana Islands",
        "Paracel Islands", "Bangladesh", "Laos", "Thailand"
    ]:
        if label == "Philippines":
            ax.text(x + 6, y + 2, label, fontsize=label_fontsize, ha='center', color='black')
        elif label == "Malaysia":
            ax.text(x - 0.5, y + 0, label, fontsize=label_fontsize, ha='center', color='black')
        elif label == "East Timor":
            ax.text(x + 1, y - 0.9, label, fontsize=label_fontsize, ha='center', color='black')
        else:
            ax.text(x, y, label, fontsize=label_fontsize, ha='center', color='black')

ax.set_ylim(lat_min, lat_max)
ax.set_xlim(lon_min, lon_max)
ax.set_xticks([])
ax.set_yticks([])
ax.text(lon_max - 1, lat_max - 3.5, "d", color="black", fontsize=16, fontweight="bold", ha="right", va="bottom")

plt.savefig(os.path.join(output_dir, 'panel_d.png'), dpi=300, bbox_inches='tight')
plt.show()


# Panel e: PACIFIC ISLANDS
merged1 = merged[merged['SUBREGION'].isin(['Polynesia', 'Melanesia', 'Micronesia'])]

# figsize chosen so saved height matches Panel c's 4-inch height
# (data extent 35 deg lon x 15 deg lat -> aspect ~2.33:1).
fig = plt.figure(figsize=(9, 4))
fig.patch.set_facecolor('none')
ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=-170))
ax.set_facecolor((0.91, 0.98, 1, 1))

merged1.plot(
    column='diff_adjusted',
    cmap=GnBu_darker2,
    norm=norm,
    edgecolor='black',
    linewidth=0.2,
    legend=False,
    transform=ccrs.PlateCarree(),
    ax=ax,
    missing_kwds={"color": "lightgrey", "alpha": 0.5, "label": "No data"},
    vmax=40
)

# Overlay: hatch territories without its own debt data
merged1.loc[merged1["NAME"] == "New Caledonia"].plot(
    ax=ax,
    facecolor="none",        # don't overwrite the colormap
    edgecolor="grey",       # hatch color comes from edgecolor
    linewidth=0.05,
    hatch="////",
    transform=ccrs.PlateCarree()
)

x_min, x_max = -35, 0
y_min, y_max = -22, -7
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

label_fontsize = 7
for x, y, label in zip(merged1.geometry.centroid.x, merged1.geometry.centroid.y, merged1['GEOUNIT']):
    if label in ["Solomon Islands", "Samoa", "Fiji", "Vanuatu", "Tonga", "Tuvalu"]:
        if label == "Samoa":
            ax.text(-2, y + 1, label, fontsize=label_fontsize, ha='center', color='black')
        elif label == "Solomon Islands":
            ax.text(-26, y + 0, "Solomon \nIslands", fontsize=label_fontsize, ha='center', color='black')
        elif label == "Fiji":
            ax.text(-12, y + 2, label, fontsize=label_fontsize, ha='center', color='black')
        elif label == "Vanuatu":
            ax.text(-26, y + 0, label, fontsize=label_fontsize, ha='center', color='black')
        elif label == "Tuvalu":
            ax.text(-12, y - 0.5, label, fontsize=label_fontsize, ha='center', color='black')
        elif label == "Tonga":
            ax.text(-5, y + 0, label, fontsize=label_fontsize, ha='center', color='black')

ax.set_xticks([])
ax.set_yticks([])
ax.text(-1, y_max - 1.9, "e", color="black", fontsize=17, fontweight="bold", ha="right", va="bottom")

plt.savefig(os.path.join(output_dir, 'panel_e.png'), dpi=300, bbox_inches='tight')
plt.show()

### Fig. 4 panels b, c, e — combined strip

In [ ]:
# combine panels b, c, and e from above

import matplotlib.image as mpimg
from matplotlib.gridspec import GridSpec

panel_dir = "./4_figures/main/M4_map"
img_b = mpimg.imread(f"{panel_dir}/panel_b.png")
img_c = mpimg.imread(f"{panel_dir}/panel_c.png")
img_e = mpimg.imread(f"{panel_dir}/panel_e.png")

h_b, w_b = img_b.shape[:2]
h_c, w_c = img_c.shape[:2]
h_e, w_e = img_e.shape[:2]

# One row: panels scaled to a common height, widths proportional to aspect.
H = max(h_b, h_c, h_e)
w_b_s = w_b * H / h_b
w_c_s = w_c * H / h_c
w_e_s = w_e * H / h_e
W_total = w_b_s + w_c_s + w_e_s

DPI = 150
fig = plt.figure(figsize=(W_total / DPI, H / DPI), dpi=DPI)
gs = GridSpec(
    1, 3, figure=fig,
    width_ratios=[w_b_s, w_c_s, w_e_s],
    hspace=0.0, wspace=0.0,
)

ax_b = fig.add_subplot(gs[0, 0]); ax_b.imshow(img_b); ax_b.axis('off')
ax_c = fig.add_subplot(gs[0, 1]); ax_c.imshow(img_c); ax_c.axis('off')
ax_e = fig.add_subplot(gs[0, 2]); ax_e.imshow(img_e); ax_e.axis('off')

plt.subplots_adjust(left=0, right=1, top=1, bottom=0, hspace=0, wspace=0)

out_path = f"{panel_dir}/panel_bce_combined.png"
plt.savefig(out_path, bbox_inches='tight', pad_inches=0, dpi=DPI)
plt.show()
print(f"Saved: {out_path}")

### Fig. 4 panel f — regional counterfactual (hybrid metric)

In [ ]:
# --- Plot ---
region_order_v = ['Global', 'SIDS', 'non-SIDS', 'Americas', 'Asia', 'Oceania']

fig, axes = plt.subplots(1, len(region_order_v),
                         figsize=(2.0 * len(region_order_v) + 1.0, 2.4),
                         sharey=True)
if len(region_order_v) == 1:
    axes = [axes]

scenarios_str_v = [str(s) for s in bar_subset_order_v]

for col, group in enumerate(region_order_v):
    if group == 'Global':
        country_sub = all_impact_v
    elif group == 'SIDS':
        country_sub = all_impact_v[all_impact_v['is_sids'] == 1]
    elif group == 'non-SIDS':
        country_sub = all_impact_v[all_impact_v['is_sids'] == 0]
    else:
        country_sub = all_impact_v[all_impact_v['region'] == group]
    n_countries = country_sub['ISO'].nunique()

    ax_val = axes[col]
    val_data = [country_sub.loc[country_sub['scenario'] == s, 'plot_rate'].dropna().values * 100
                for s in scenarios_str_v]
    bp = ax_val.boxplot(val_data, patch_artist=True, showfliers=True,
                        medianprops={'color': 'black', 'linewidth': 1.2},
                        flierprops={'marker': '.', 'markersize': 4,
                                     'markerfacecolor': 'gray', 'markeredgecolor': 'gray'},
                        widths=0.6)
    for patch, color in zip(bp['boxes'], subset_colors_v):
        patch.set_facecolor(color); patch.set_alpha(0.85); patch.set_edgecolor('white')
    ax_val.set_xticks(range(1, len(scenarios_str_v) + 1))
    ax_val.set_xticklabels(scenarios_str_v, fontsize=7)
    ax_val.set_yticks([0, 50, 100, 150])
    ax_val.tick_params(axis='y', labelsize=7)
    ax_val.text(0.97, 0.96, f'{group}\n(n={n_countries})',
                transform=ax_val.transAxes, ha='right', va='top',
                fontsize=9, fontweight='bold')
    ax_val.grid(axis='y', alpha=0.2)
    ax_val.spines[['top', 'right']].set_visible(False)
    if col == 0:
        ax_val.set_ylabel('% of debt ratio\nattributable to TCs', fontsize=7.5)

plt.tight_layout()
plt.subplots_adjust(wspace=0.12, top=0.88)
fig.text(0.005, 0.97, 'f', ha='left', va='top', fontsize=18, fontweight='bold')
os.makedirs('./4_figures/main/M4_map', exist_ok=True)
plt.savefig('./4_figures/main/M4_map/regional_counterfactual_hybrid.pdf',
            bbox_inches='tight')
plt.show()

### Fig. 5 (two parts)

Driven by `2_code/main/figures/M5_basispoint_plot.py`, which reads the 500-seed XGB predictions (committed as five seed-range parts) and writes `M5_partA_grade.png` + `M5_partB_nonrated.png`.

In [ ]:
import subprocess, sys
_r = subprocess.run(
    [sys.executable, 'M5_basispoint_plot.py', '--seeds', '500', '--all-isos', '--metric', 'bps'],
    cwd='2_code/main/figures', capture_output=True, text=True)
print(_r.stdout[-1200:] or _r.stderr[-1200:])
if _r.returncode != 0:
    print('M5 driver exited', _r.returncode)

# Show the two saved Fig. 5 panels (M5_partA_grade = investment + speculative grade,
# M5_partB_nonrated = non-rated countries).
from IPython.display import Image, display
for _f in ('M5_partA_grade.png', 'M5_partB_nonrated.png'):
    display(Image(filename=os.path.join('4_figures', 'main', 'M5_basispoint', _f), width=900))
